# KD figure

Control vs shAPLNR gastruloids. Panels in narrative order: knockdown works → composition shifts → fate arrest → WNT/module collapse → migration programme → bulk convergence. Full exploratory analysis (depth controls, FOXC2 diagnostics, genome-wide scans) lives in FIGURE_KD_CLEAN as the provenance record.

## Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
plt.rcParams["pdf.fonttype"] = 42

import sys
sys.path.insert(0, str(Path.cwd().parent / "utils"))   # run notebooks from their own folder
from config import ROOT, PROJECT_DIR, CELLRANGER_DIR
MASTER   = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized_magic.h5ad"
STAN_CSV = ROOT / "00_DATA_FROZEN" / "01_source" / "sample_barcodes.csv"
FIG = ROOT / "07_figures" / "FIGURE_KD_clean"; FIG.mkdir(parents=True, exist_ok=True)
TAB = ROOT / "06_tables"  / "FIGURE_KD_clean"; TAB.mkdir(parents=True, exist_ok=True)

GENES = ["APLNR", "APELA"]
TIMES = ["42h", "72h"]
SAVE_FORMATS = ["svg", "pdf"]
SHOW_TITLES  = False
MATCH_STAN   = True     # single filter; set False to use all QC-passing cells

RED, BLUE  = "#991E03", "#6B8E9E"
GREY_FAINT = "#F2EFEC"
SITE_COLORS = {"Michigan": "#4A7C7E", "Chicago": "#D98E5A"}
DOT_CMAP   = LinearSegmentedColormap.from_list("dot", [GREY_FAINT, RED])
# TISSUE_COLORS = {
#     "pluripotent":"#6B8E9E","AE-LC":"#8B6A9E","aPS-LC":"#4A7C7E","pPS-LC":"#B8A88A",
#     "NasM":"#C0563E","DE":"#D4A94E","PGC-LC":"#BE7F96","other":"#AFA9A4",
# }

def thin_size_legend(dp, keep=3):
    try:
        sl = dp.get_axes().get("size_legend_ax")
        if sl is None or not sl.collections: return
        coll = sl.collections[0]; sizes = np.asarray(coll.get_sizes())
        labels = [t.get_text() for t in sl.get_xticklabels()]
        colors = coll.get_facecolors(); n = len(sizes)
        if n <= keep: sl.grid(False); return
        idx = np.linspace(0, n-1, keep).round().astype(int)
        color = colors[0] if len(colors) else "gray"
        sl.clear()
        sl.scatter(np.arange(len(idx)), np.zeros(len(idx)), s=sizes[idx],
                   color=color, edgecolor="black", linewidth=0.25, zorder=100)
        sl.set_xticks(np.arange(len(idx)))
        if len(labels)==n: sl.set_xticklabels([labels[i] for i in idx], fontsize="small")
        sl.set_yticks([]); sl.grid(False)
        sl.tick_params(axis="both", left=False, bottom=False, labelleft=False)
        for s in sl.spines.values(): s.set_visible(False)
        sl.set_xlim(-0.7, len(idx)-0.3); sl.set_ylim(-0.5, 0.5)
    except Exception as e: print(f"  (size legend skip: {e})")

print("inputs:", MASTER.exists(), STAN_CSV.exists())

In [ ]:
from palettes import (TISSUE_COLORS, SITE_COLORS, PROGRAM_COLORS,
                      TISSUE_MAP, TISSUE_ORDER)
TISSUE_COLORS["aPM-LC"] = "#D98270"
if "aPM-LC" not in TISSUE_ORDER: TISSUE_ORDER.append("aPM-LC")

LEGEND_FS = 10
SHOW_TITLES = True

RED, BLUE  = "#991E03", "#6B8E9E"
GREY_FAINT = "#F2EFEC"

adata = ad.read_h5ad(MASTER)
adata.obs["tissue"] = pd.Categorical(
    adata.obs["predicted_leiden_sub_joint"].astype(str).map(TISSUE_MAP),
    categories=TISSUE_ORDER, ordered=True)
adata.obs["cond"] = adata.obs["condition"].astype(str).map({"Ctrl":"Ctrl","DOX":"KD"})
adata.obs["sample"] = (adata.obs["site"].astype(str)+"_"+adata.obs["cond"].astype(str)+
                       "_"+adata.obs["timepoint_true"].astype(str))

def bc16(x):
    h = re.findall(r"[ACGT]{14,}", str(x)); return h[0] if h else None
if MATCH_STAN:
    stan = pd.read_csv(STAN_CSV, index_col=0)
    STAN_BC = set(bc16(x) for x in stan.index)
    adata.obs["_bc"] = [bc16(x) for x in adata.obs_names]
    adata = adata[adata.obs["_bc"].isin(STAN_BC)].copy()
print(f"cells: {adata.n_obs:,}  (MATCH_STAN={MATCH_STAN})")
print(adata.obs.groupby(["site","cond","timepoint_true"], observed=True).size().to_string())

sym2key = dict(zip(adata.var["symbol"].astype(str), adata.var.index)) \
          if "symbol" in adata.var else {g:g for g in adata.var_names}
GK = {g: sym2key[g] for g in GENES if g in sym2key}

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
TISSUE_ORDER = [
    'pluripotent', 'AE-LC', 'pPS-LC', 'aPS-LC',
    'eNasM', 'lNasM', 'DE', 'PGC-LC',
    'aPM-LC', 'CS-LC', 'LPM-LC', 'stressed',
]
adata.obs["tissue"] = pd.Categorical(
    adata.obs["predicted_leiden_sub_joint"].astype(str),
    categories=TISSUE_ORDER, ordered=True)
def side_legend(ax, labels, colors, **kw):
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=colors[l], edgecolor="none", label=l) for l in labels]
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=8, **kw)

TISSUE_COLORS["aPM-LC"] = "#D98270"
if "aPM-LC" not in TISSUE_ORDER: TISSUE_ORDER.append("aPM-LC")

xy    = np.asarray(adata.obsm["X_umap_joint"])
GROUPS  = [("Ctrl","42h"), ("Ctrl","72h"), ("KD","42h"), ("KD","72h")]
GLABELS = ["Ctrl 42h", "Ctrl 72h", "KD 42h", "KD 72h"]
cats  = [t for t in TISSUE_ORDER if t in adata.obs["tissue"].cat.categories]

# ── UMAPs ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(28, 6.5))
for ax, (cond, tp), label in zip(axes, GROUPS, GLABELS):
    m = ((adata.obs["cond"].astype(str) == cond) &
         (adata.obs["timepoint_true"].astype(str) == tp)).to_numpy()
    for t in cats:
        mt = m & (adata.obs["tissue"].astype(str) == t).to_numpy()
        if mt.sum(): ax.scatter(xy[mt,0], xy[mt,1], s=20,
                                color=TISSUE_COLORS[t], linewidths=0, rasterized=True)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    ax.set_aspect("equal"); ax.grid(False)
    if SHOW_TITLES: ax.set_title(label, fontsize=LEGEND_FS + 2)
    ax.text(0.02, 0.02, f"n={m.sum():,}", transform=ax.transAxes, fontsize=8, color="grey")
side_legend(axes[-1], cats, TISSUE_COLORS)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"2_umap_ctrl_kd.{ext}", bbox_inches="tight")
plt.show()

# ── Stacked bars ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 5.5))
bottom = np.zeros(4)
for t in cats:
    vals = []
    for cond, tp in GROUPS:
        m = ((adata.obs["cond"].astype(str) == cond) &
             (adata.obs["timepoint_true"].astype(str) == tp)).to_numpy()
        vals.append(((adata.obs["tissue"].astype(str) == t) & m).sum() / m.sum()
                    if m.sum() > 0 else 0)
    ax.bar(np.arange(4), vals, bottom=bottom,
           color=TISSUE_COLORS[t], width=0.6, edgecolor="white", linewidth=0.8)
    bottom += np.array(vals)

ax.set_xticks(np.arange(4))
ax.set_xticklabels(GLABELS, fontsize=LEGEND_FS)
ax.set_ylabel("fraction of cells", fontsize=13)
ax.set_ylim(0, 1); ax.grid(False)
ax.spines[["top","right"]].set_visible(False)
side_legend(ax, cats, TISSUE_COLORS)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"2_composition_ctrl_kd.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.sparse as sp

GENES = ["APLNR", "APELA", "APLN"]

counts = adata.layers["counts"]
lib    = np.asarray(counts.sum(axis=1)).ravel()

rows = []
for s in adata.obs["sample"].unique():
    m = (adata.obs["sample"] == s).to_numpy()
    total_umi = float(lib[m].sum())
    rec = {"sample":    s,
           "site":      adata.obs.loc[m, "site"].iloc[0],
           "cond":      adata.obs.loc[m, "cond"].iloc[0],
           "timepoint": adata.obs.loc[m, "timepoint_true"].iloc[0],
           "n_cells":   int(m.sum())}
    for g in GENES:
        j = adata.var_names.get_loc(GK[g])
        raw = counts[m, j]
        rec[f"{g}_cpm"] = float(np.asarray(raw.todense()).sum()
                                if sp.issparse(raw) else raw.sum()) / total_umi * 1e6
    rows.append(rec)

pb = pd.DataFrame(rows).sort_values(["site", "timepoint", "cond"])
pb.to_csv(TAB / "1_kd_validation.csv", index=False)
display(pb[["sample", "n_cells", "APLNR_cpm", "APELA_cpm", "APLN_cpm"]].round(3))

fig, axes = plt.subplots(1, len(GENES), figsize=(len(GENES)*3.2, 4))
for ax, g in zip(axes, GENES):
    for site in sorted(pb["site"].unique()):
        for tp in TIMES:
            d = pb[(pb.site==site) & (pb.timepoint==tp)].sort_values("cond")
            if len(d) < 2: continue
            ax.plot([0, 1], d[f"{g}_cpm"].values,
                    marker="o", color=SITE_COLORS[site],
                    ls="-" if tp=="42h" else "--",
                    label=f"{site} {tp}", ms=5, lw=1.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Ctrl", "KD"], fontsize=10)
    ax.set_ylabel(f"{g} CPM (pseudobulk)", fontsize=10)
    if SHOW_TITLES: ax.set_title(g)
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG / f"1_kd_validation.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(GENES), figsize=(len(GENES)*3, 6), sharey=False)

for col, g in enumerate(GENES):
    for row, tp in enumerate(TIMES):
        ax = axes[row, col]
        sub = pb[pb["timepoint"] == tp].sort_values("cond")
        ctrl_vals = sub[sub["cond"]=="Ctrl"][f"{g}_cpm"].values
        kd_vals   = sub[sub["cond"]=="KD"][f"{g}_cpm"].values
        sites     = sub[sub["cond"]=="Ctrl"]["site"].values

        ax.bar([0,1], [ctrl_vals.mean(), kd_vals.mean()],
               color=["#2166ac","#d6604d"], width=0.5, alpha=0.7, zorder=2)
        for s_i, site in enumerate(sites):
            c = SITE_COLORS[site]
            ax.scatter([0, 1], [ctrl_vals[s_i], kd_vals[s_i]],
                       color=c, s=40, zorder=3, edgecolors='white', linewidths=0.8)
            ax.plot([0, 1], [ctrl_vals[s_i], kd_vals[s_i]],
                    color=c, lw=0.8, alpha=0.5, zorder=2)

        ax.set_xticks([0,1]); ax.set_xticklabels(["Ctrl","KD"], fontsize=9)
        ax.spines[["top","right"]].set_visible(False)
        if col == 0: ax.set_ylabel(f"{tp}\nCPM (pseudobulk)", fontsize=9)
        if row == 0: ax.set_title(g, fontsize=10)

fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"1_kd_validation.{ext}", bbox_inches="tight")
plt.show()

## A — knockdown validation (pseudobulk APLNR per sample)

In [ ]:
counts = adata.layers["counts"]
lib = np.asarray(counts.sum(axis=1)).ravel()
rows=[]
for s in adata.obs["sample"].unique():
    m = (adata.obs["sample"]==s).to_numpy()
    rec={"sample":s,"site":adata.obs.loc[m,"site"].iloc[0],
         "cond":adata.obs.loc[m,"cond"].iloc[0],
         "timepoint":adata.obs.loc[m,"timepoint_true"].iloc[0],
         "n_cells":int(m.sum()),"total_umi":float(lib[m].sum())}
    for g in GENES:
        j = adata.var_names.get_loc(GK[g])
        rec[f"{g}_cpm"]=float(counts[m,j].sum()/rec["total_umi"]*1e6)
    rows.append(rec)
pb = pd.DataFrame(rows).sort_values(["site","timepoint","cond"])
pb.to_csv(TAB/"1_kd_validation.csv", index=False)
display(pb[["sample","n_cells","APLNR_cpm","APELA_cpm"]].round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax,g in zip(axes, GENES):
    for i,site in enumerate(sorted(pb["site"].unique())):
        for t in TIMES:
            d = pb[(pb.site==site)&(pb.timepoint==t)].sort_values("cond", ascending=False)
            if len(d)<2: continue
            ax.plot([0,1], d[f"{g}_cpm"], marker="o", color=SITE_COLORS[site],
                    ls="-" if t=="42h" else "--", label=f"{site} {t}")
    ax.set_xticks([0,1]); ax.set_xticklabels(["Ctrl","KD"])
    ax.set_ylabel(f"{g} (CPM, pseudobulk)")
    if SHOW_TITLES: ax.set_title(g)
    ax.spines[["top","right"]].set_visible(False)
axes[1].legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.02,0.5))
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"1_kd_validation.{ext}", bbox_inches="tight")
plt.show()

## B — tissue composition, Ctrl vs KD

In [ ]:
comp = (adata.obs.groupby(["site","cond","timepoint_true","tissue"], observed=True)
        .size().rename("n").reset_index())
comp["fraction"] = comp["n"]/comp.groupby(["site","cond","timepoint_true"])["n"].transform("sum")
comp.to_csv(TAB/"2_composition.csv", index=False)

sites = sorted(adata.obs["site"].astype(str).unique())
groups = [(s,t,c) for s in sites for t in TIMES for c in ["Ctrl","KD"]]
cats = TISSUE_ORDER
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(groups))
for c in cats:
    vals = [float(comp[(comp.site==s)&(comp.timepoint_true==t)&(comp.cond==cd)&
                       (comp.tissue==c)]["fraction"].sum()) for s,t,cd in groups]
    ax.bar(np.arange(len(groups)), vals, bottom=bottom, color=TISSUE_COLORS[c],
           width=0.7, edgecolor="white", linewidth=0.8)
    bottom += np.array(vals)
ax.set_xticks(np.arange(len(groups)))
ax.set_xticklabels([f"{s[:4]}\n{t}\n{cd}" for s,t,cd in groups], fontsize=8)
ax.set_ylabel("fraction of cells"); ax.set_ylim(0,1); ax.grid(False)
h=[plt.Line2D([0],[0],marker="o",ls="",markersize=9,markerfacecolor=TISSUE_COLORS[c],
              markeredgecolor="none",label=c) for c in cats]
ax.legend(handles=h, loc="center left", bbox_to_anchor=(1.02,0.5), frameon=False, fontsize=11)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"2_composition.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
comp_combined = (
    comp
    .groupby(
        ["site", "cond", "timepoint_true", "tissue"],
        observed=True,
    )["fraction"]
    .sum()
    .reset_index()
    .groupby(
        ["cond", "timepoint_true", "tissue"],
        observed=True,
    )["fraction"]
    .mean()
    .reset_index()
)

groups = [
    ("42h", "Ctrl"),
    ("42h", "KD"),
    ("72h", "Ctrl"),
    ("72h", "KD"),
]

# Exact tissue fractions plotted in each bar
group_index = pd.MultiIndex.from_tuples(
    groups,
    names=["timepoint_true", "cond"],
)

fraction_table = (
    comp_combined
    .pivot(
        index=["timepoint_true", "cond"],
        columns="tissue",
        values="fraction",
    )
    .reindex(index=group_index, columns=TISSUE_ORDER)
    .fillna(0)
)

print("\nMean fractions:")
print(fraction_table.round(4).to_string())

print("\nMean percentages:")
print((fraction_table * 100).round(1).to_string())

print("\nColumn totals:")
print(fraction_table.sum(axis=1).round(4).to_string())

fig, ax = plt.subplots(figsize=(5.5, 5))

bottom = np.zeros(len(groups))

for tissue in TISSUE_ORDER:

    vals = []

    for timepoint, condition in groups:

        hit = comp_combined[
            (comp_combined["timepoint_true"].astype(str) == timepoint)
            & (comp_combined["cond"].astype(str) == condition)
            & (comp_combined["tissue"].astype(str) == tissue)
        ]

        vals.append(
            float(hit["fraction"].iloc[0])
            if len(hit) > 0
            else 0
        )

    ax.bar(
        np.arange(len(groups)),
        vals,
        bottom=bottom,
        color=TISSUE_COLORS[tissue],
        width=0.72,
        edgecolor="white",
        linewidth=0.8,
    )

    bottom += np.asarray(vals)

ax.set_xticks(np.arange(len(groups)))
ax.set_xticklabels(
    [f"{t}\n{c}" for t, c in groups]
)

ax.set_ylabel("Mean fraction of cells")
ax.set_ylim(0, 1)
ax.grid(False)

ax.spines[["top", "right"]].set_visible(False)

handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=8,
        markerfacecolor=TISSUE_COLORS[tissue],
        markeredgecolor="none",
        label=tissue,
    )
    for tissue in TISSUE_ORDER
]

ax.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

fig.tight_layout()

for ext in SAVE_FORMATS:
    fig.savefig(
        FIG / f"2_composition_sites_combined.{ext}",
        bbox_inches="tight",
    )

plt.show()

In [ ]:
for label, cond_val in [('Ctrl', 'Ctrl'), ('KD', 'KD')]:
    for tp in ['42h', '72h']:
        sub = adata[(adata.obs['cond'].astype(str) == cond_val) &
                    (adata.obs['timepoint_true'].astype(str) == tp)]
        total = sub.n_obs
        counts = sub.obs['tissue'].value_counts()
        print(f"\n{label} {tp}: n={total:,}")
        for tissue, n in counts.items():
            print(f"  {tissue}: {n} ({n/total*100:.1f}%)")

In [ ]:
# exact values used in the composition bar plot
for cond in ["Ctrl", "KD"]:

    print(f"\n{cond} — 72h")

    d = comp_combined[
        (comp_combined["timepoint_true"].astype(str) == "72h")
        & (comp_combined["cond"].astype(str) == cond)
    ]

    e = d.loc[
        d["tissue"].astype(str) == "eNasM",
        "fraction"
    ].iloc[0]

    l = d.loc[
        d["tissue"].astype(str) == "lNasM",
        "fraction"
    ].iloc[0]

    nasm = e + l

    print(f"eNasM: {100*e:.2f}% of all cells")
    print(f"lNasM: {100*l:.2f}% of all cells")
    print(f"NasM total: {100*nasm:.2f}% of all cells")

    print(f"within NasM: eNasM = {100*e/nasm:.2f}%")
    print(f"within NasM: lNasM = {100*l/nasm:.2f}%")
    print(f"eNasM:lNasM = {e/l:.2f}")
    
x = comp[
    (comp["timepoint_true"].astype(str) == "72h")
    & (comp["tissue"].astype(str).isin(["eNasM", "lNasM"]))
].copy()

print(
    x.pivot_table(
        index=["site", "cond"],
        columns="tissue",
        values="fraction",
        aggfunc="sum"
    )
    .assign(
        NasM=lambda d: d["eNasM"] + d["lNasM"],
        e_to_l=lambda d: d["eNasM"] / d["lNasM"],
        l_within_NasM=lambda d: d["lNasM"] / (d["eNasM"] + d["lNasM"])
    )
    .round(3)
)


In [ ]:
# ============================================================================
# APLNR KD temporal mesoderm dot plot
#
# EXACT developmental groups shown:
#   42h aPS-LC
#   42h pPS-LC
#   72h eNasM
#   72h lNasM
#
# Within each group:
#   Ctrl
#   KD
#
# Dot size  = fraction of cells expressing
# Dot color = scaled expression (Reds)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc


# ---------------------------------------------------------------------------
# 1. Exact state × time combinations to show
# ---------------------------------------------------------------------------

MESO_GROUPS = [
    ("42h", "aPS-LC", "aPS 42h"),
    ("42h", "pPS-LC", "pPS 42h"),
    ("72h", "eNasM",  "eNasM 72h"),
    ("72h", "lNasM",  "lNasM 72h"),
]

MIN_CELLS_ROW = 30


# ---------------------------------------------------------------------------
# 2. Genes
#
# This ordering is intentional:
# early signaling -> mesoderm/PSM -> ECM -> alternative state
# ---------------------------------------------------------------------------

MESO_MARKERS = {
    
    "BMP response": [
        "ID1",
        "ID3",
        "MSX2",
    ],

    "early meso.": [
        "TBXT",
        "MIXL1",
        "MESP1",
        # "NODAL",   # optional
    ],
    
    "morphogenesis": [
        "SNAI1",
        "SPARC",
        "VIM",
    ],
        
    "late patterning": [
        "RSPO3",
        "DLL3",
        "TBX6",
        "WNT8A",
        "SP5",
        "MESP2",
        "FOXC1",
    ],



}

# ---------------------------------------------------------------------------
# 3. Resolve gene symbols -> adata var keys
# ---------------------------------------------------------------------------

if "symbol" in adata.var.columns:

    _sym2key = (
        adata.var
        .reset_index()
        .drop_duplicates("symbol")
        .set_index("symbol")["index"]
        .astype(str)
        .to_dict()
    )

else:

    _sym2key = {
        str(g): str(g)
        for g in adata.var_names
    }


_key2sym = {
    v: k
    for k, v in _sym2key.items()
}


def _resolve(mapping):

    out = {}
    missing = []

    for grp, genes in mapping.items():

        keys = [
            _sym2key[g]
            for g in genes
            if g in _sym2key
        ]

        missing += [
            g
            for g in genes
            if g not in _sym2key
        ]

        if keys:
            out[grp] = keys

    if missing:
        print(
            "Not in data:",
            ", ".join(missing)
        )

    return out


marks = _resolve(
    MESO_MARKERS
)


# ---------------------------------------------------------------------------
# 4. Select ONLY the desired time × state combinations
# ---------------------------------------------------------------------------

pair_mask = np.zeros(
    adata.n_obs,
    dtype=bool,
)


for t, state_name, _ in MESO_GROUPS:

    pair_mask |= (
        (
            adata.obs[
                "timepoint_true"
            ].astype(str)
            == t
        )
        &
        (
            adata.obs[
                "predicted_leiden_sub_joint"
            ].astype(str)
            == state_name
        )
    ).to_numpy()


sub = adata[
    pair_mask
].copy()


# ---------------------------------------------------------------------------
# 5. Standardize condition labels
# ---------------------------------------------------------------------------

sub.obs[
    "_plot_condition"
] = (
    sub.obs[
        "condition"
    ]
    .astype(str)
    .map(
        {
            "Ctrl": "Ctrl",
            "DOX": "KD",
        }
    )
)


# ---------------------------------------------------------------------------
# 6. Determine which state/time groups have enough cells
#    in BOTH Ctrl and KD
# ---------------------------------------------------------------------------

usable_groups = []

print(
    "\nCell counts:"
)


for t, state_name, pretty_name in MESO_GROUPS:

    m = (
        (
            sub.obs[
                "timepoint_true"
            ].astype(str)
            == t
        )
        &
        (
            sub.obs[
                "predicted_leiden_sub_joint"
            ].astype(str)
            == state_name
        )
    )

    tab = (
        sub.obs.loc[
            m,
            "_plot_condition"
        ]
        .value_counts()
    )

    n_ctrl = int(
        tab.get(
            "Ctrl",
            0,
        )
    )

    n_kd = int(
        tab.get(
            "KD",
            0,
        )
    )

    print(
        f"{pretty_name:12s}  "
        f"Ctrl={n_ctrl:,}   "
        f"KD={n_kd:,}"
    )

    if (
        n_ctrl >= MIN_CELLS_ROW
        and
        n_kd >= MIN_CELLS_ROW
    ):

        usable_groups.append(
            (
                t,
                state_name,
                pretty_name,
            )
        )

    else:

        print(
            f"  -> dropped: fewer than "
            f"{MIN_CELLS_ROW} cells "
            f"in one condition"
        )


if not usable_groups:

    raise ValueError(
        "No state/time groups pass "
        "MIN_CELLS_ROW."
    )


# ---------------------------------------------------------------------------
# 7. Create exact row labels
# ---------------------------------------------------------------------------

row_values = []

keep_mask = np.zeros(
    sub.n_obs,
    dtype=bool,
)


for t, state_name, pretty_name in usable_groups:

    m = (
        (
            sub.obs[
                "timepoint_true"
            ].astype(str)
            == t
        )
        &
        (
            sub.obs[
                "predicted_leiden_sub_joint"
            ].astype(str)
            == state_name
        )
    )

    keep_mask |= m.to_numpy()


sub = sub[
    keep_mask
].copy()


# Cell counts for labels
count_lookup = {}


for t, state_name, pretty_name in usable_groups:

    for cond in [
        "Ctrl",
        "KD",
    ]:

        m = (
            (
                sub.obs[
                    "timepoint_true"
                ].astype(str)
                == t
            )
            &
            (
                sub.obs[
                    "predicted_leiden_sub_joint"
                ].astype(str)
                == state_name
            )
            &
            (
                sub.obs[
                    "_plot_condition"
                ].astype(str)
                == cond
            )
        )

        count_lookup[
            (
                t,
                state_name,
                cond,
            )
        ] = int(
            m.sum()
        )


# Explicit display order
row_order = []


for t, state_name, pretty_name in usable_groups:

    for cond in [
        "Ctrl",
        "KD",
    ]:

        n = count_lookup[
            (
                t,
                state_name,
                cond,
            )
        ]

        row_order.append(
            f"{pretty_name}  "
            f"{cond} "
            f"(n={n:,})"
        )


# Assign row string per cell
row_series = pd.Series(
    index=sub.obs_names,
    dtype="object",
)


for t, state_name, pretty_name in usable_groups:

    for cond in [
        "Ctrl",
        "KD",
    ]:

        m = (
            (
                sub.obs[
                    "timepoint_true"
                ].astype(str)
                == t
            )
            &
            (
                sub.obs[
                    "predicted_leiden_sub_joint"
                ].astype(str)
                == state_name
            )
            &
            (
                sub.obs[
                    "_plot_condition"
                ].astype(str)
                == cond
            )
        )

        n = count_lookup[
            (
                t,
                state_name,
                cond,
            )
        ]

        row_series.loc[
            sub.obs_names[
                m
            ]
        ] = (
            f"{pretty_name}  "
            f"{cond} "
            f"(n={n:,})"
        )


sub.obs[
    "meso_plot_row"
] = pd.Categorical(
    row_series,
    categories=row_order,
    ordered=True,
)


# ---------------------------------------------------------------------------
# 8. Plot
# ---------------------------------------------------------------------------

n_genes = sum(
    len(v)
    for v in marks.values()
)


dp = sc.pl.dotplot(

    sub,

    marks,

    groupby="meso_plot_row",

    categories_order=row_order,

    use_raw=False,

    # IMPORTANT:
    # color is scaled independently for each gene
    # across the displayed groups
    standard_scale="var",

    cmap="Reds",

    dot_max=1.0,

    dot_min=0.0,

    smallest_dot=14,

    var_group_rotation=45,

    figsize=(
        0.32 * n_genes + 2.3,
        0.15 * len(row_order) + 1.6,
    ),

    title=None,

    return_fig=True,
)


dp.style(

    dot_edge_color="black",

    dot_edge_lw=0.25,

    largest_dot=220,
)


dp.legend(

    width=1.7,

    colorbar_title=(
        "Scaled expression"
    ),

    size_title=(
        "Fraction of cells (%)"
    ),
)


dp.make_figure()


# ---------------------------------------------------------------------------
# 9. Clean x-axis gene labels
# ---------------------------------------------------------------------------

try:

    mp = dp.get_axes()[
        "mainplot_ax"
    ]

    old_labels = [
        tick.get_text()
        for tick
        in mp.get_xticklabels()
    ]

    clean_labels = [
        _key2sym.get(
            label,
            label,
        )
        for label
        in old_labels
    ]

    mp.set_xticklabels(

        clean_labels,

        rotation=70,

        ha="right",
    )


except Exception as e:

    print(
        f"x-label cleanup skipped: {e}"
    )


# ---------------------------------------------------------------------------
# 10. Save
# ---------------------------------------------------------------------------

for ext in [
    "svg",
    "pdf",
]:

    dp.fig.savefig(

        FIG
        / f"KD_temporal_PS_to_NasM_dotplot.{ext}",

        bbox_inches="tight",
    )


plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Mesoderm-focused dot plot, Ctrl vs KD.
# Rows are state x condition so the two sit adjacent and the shift is readable.
# Paste into FIGURE_KD.ipynb after the config cell.
# ---------------------------------------------------------------------------

MESO_STATES = ["aPS-LC", "pPS-LC"]
MIN_CELLS_ROW = 5        # states below this in either condition are dropped
MESO_TIMES    = ["42h"]   # add "42h" if you want both

MESO_MARKERS = {
    "apelin receptor": ["APLNR"],

    "BMP response": [
        "MSX2", "ID3", "ID1"
    ],

    "early mesoderm": [
        "MESP1", "LHX1", "MIXL1", "TBXT"
    ],

    "WNT / late patterning": [
        "WNT8A", "FZD2", "SP5", "RSPO3", "FGF17",
        "DLL3", "TBX6", "MESP2", "EVX1", "FOXC1", "FOXC2"
    ],

    "morphogenesis": [
        "GJA1", "VIM", "SERPINE2",
        "SNAI1", "SPARC", "CITED2"
    ],
}

# MESO_MARKERS = {
#     "streak":      ["TBXT", "MIXL1", "EOMES"],
#     "nascent/PSM": ["MESP1", "MESP2", "MSGN1", "TBX6", "DLL3", "RIPPLY2"],
#     "WNT output":  ["SP5", "RSPO3", "NKD1", "CDX2", "DKK1", "WNT8A", "AXIN2"],
#     "BMP output":  ["ID1", "ID2", "ID3", "MSX2"],
#     "LPM":         ["HAND1", "BMP4", "FOXF1", "HAND2", "PRRX1"],
#     "somite":      ["FOXC2"],            # 4.8% — keep only because you're studying it
#     "apelin":      ["APLNR", "APELA"],
# }

_sym2key = (dict(zip(adata.var["symbol"].astype(str), adata.var.index))
            if "symbol" in adata.var else {g: g for g in adata.var_names})

def _resolve(mapping):
    out, missing = {}, []
    for grp, genes in mapping.items():
        keys = [_sym2key[g] for g in genes if g in _sym2key]
        missing += [g for g in genes if g not in _sym2key]
        if keys:
            out[grp] = keys
    if missing:
        print("  not in data:", ", ".join(missing))
    return out

for t in MESO_TIMES:
    sub = adata[(adata.obs["timepoint_true"].astype(str) == t) &
                (adata.obs["predicted_leiden_sub_joint"].astype(str).isin(MESO_STATES))].copy()
    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})

    # keep only states with enough cells in BOTH conditions
    tab = pd.crosstab(state, cond)
    usable = [s for s in MESO_STATES
              if s in tab.index and tab.loc[s].min() >= MIN_CELLS_ROW]
    dropped = {s: dict(tab.loc[s]) for s in tab.index if s not in usable}
    print(f"--- {t} ---")
    if dropped:
        print(f"  dropped (under {MIN_CELLS_ROW} in one condition): {dropped}")
    if not usable:
        print("  nothing passes the threshold"); continue

    sub = sub[state.isin(usable).to_numpy()].copy()
    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})
    n = pd.crosstab(state, cond)

    order = [f"{s}  {c}" for s in usable for c in ["Ctrl", "KD"]]
    labels = {f"{s}  {c}": f"{s}  {c} (n={int(n.loc[s, c]):,})"
              for s in usable for c in ["Ctrl", "KD"]}
    sub.obs["row"] = pd.Categorical((state + "  " + cond).map(labels),
                                    categories=[labels[o] for o in order], ordered=True)

    marks = _resolve(MESO_MARKERS)
    n_genes = sum(len(v) for v in marks.values())

    dp = sc.pl.dotplot(
        sub, marks, groupby="row", categories_order=[labels[o] for o in order],
        use_raw=False, standard_scale="var",
        dot_max=1.0, dot_min=0.0, smallest_dot=14,
        #cmap=LinearSegmentedColormap.from_list("m", [GREY_FAINT, RED]),
        cmap="Reds",
        var_group_rotation=45,
        figsize=(0.30 * n_genes + 2.0, 0.30 * len(order) + 1.4),
        title=None, return_fig=True)
    dp.style(dot_edge_color="black", dot_edge_lw=0.25, largest_dot=190)
    dp.legend(width=1.6, colorbar_title="Scaled expression",
              size_title="Fraction of cells (%)")
    dp.make_figure()
    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels([g.get_text().split(" (")[0] for g in mp.get_xticklabels()],
                           rotation=70, ha="right")
    except Exception as e:
        print(f"  (x-label cleanup skipped: {e})")
    for ext in ["svg", "pdf"]:
        dp.fig.savefig(FIG / f"KD_lnasmelasmmesoderm_dotplot_{t}.{ext}", bbox_inches="tight")
    plt.show()
_sym2key = (dict(zip(adata.var["symbol"].astype(str), adata.var.index))
            if "symbol" in adata.var else {g: g for g in adata.var_names})


In [ ]:
# ---------------------------------------------------------------------------
# Mesoderm-focused dot plot, Ctrl vs KD.
# Rows are state x condition so the two sit adjacent and the shift is readable.
# Paste into FIGURE_KD.ipynb after the config cell.
# ---------------------------------------------------------------------------

#MESO_STATES = ["aPS-LC", "pPS-LC", "eNasM", "lNasM", "aPM-LC", "LPM-LC"]
MESO_STATES = ["eNasM", "lNasM", "aPM-LC", "LPM-LC"]
MIN_CELLS_ROW = 5        # states below this in either condition are dropped
MESO_TIMES    = ["72h"]   # add "42h" if you want both

MESO_MARKERS = {
    "apelin receptor": ["APLNR"],

    "BMP response": [
        "MSX2", "ID3", "ID1"
    ],

    "early mesoderm": [
        "MESP1", "LHX1", "MIXL1", "TBXT"
    ],

    "WNT / late patterning": [
        "WNT8A", "FZD2", "SP5", "RSPO3", "FGF17",
        "DLL3", "TBX6", "MESP2", "EVX1", "FOXC1", "FOXC2"
    ],

    "morphogenesis": [
        "GJA1", "VIM", "SERPINE2",
        "SNAI1", "SPARC", "CITED2"
    ],
}

# MESO_MARKERS = {
#     "streak":      ["TBXT", "MIXL1", "EOMES"],
#     "nascent/PSM": ["MESP1", "MESP2", "MSGN1", "TBX6", "DLL3", "RIPPLY2"],
#     "WNT output":  ["SP5", "RSPO3", "NKD1", "CDX2", "DKK1", "WNT8A", "AXIN2"],
#     "BMP output":  ["ID1", "ID2", "ID3", "MSX2"],
#     "LPM":         ["HAND1", "BMP4", "FOXF1", "HAND2", "PRRX1"],
#     "somite":      ["FOXC2"],            # 4.8% — keep only because you're studying it
#     "apelin":      ["APLNR", "APELA"],
# }

_sym2key = (dict(zip(adata.var["symbol"].astype(str), adata.var.index))
            if "symbol" in adata.var else {g: g for g in adata.var_names})

def _resolve(mapping):
    out, missing = {}, []
    for grp, genes in mapping.items():
        keys = [_sym2key[g] for g in genes if g in _sym2key]
        missing += [g for g in genes if g not in _sym2key]
        if keys:
            out[grp] = keys
    if missing:
        print("  not in data:", ", ".join(missing))
    return out

for t in MESO_TIMES:
    sub = adata[(adata.obs["timepoint_true"].astype(str) == t) &
                (adata.obs["predicted_leiden_sub_joint"].astype(str).isin(MESO_STATES))].copy()
    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})

    # keep only states with enough cells in BOTH conditions
    tab = pd.crosstab(state, cond)
    usable = [s for s in MESO_STATES
              if s in tab.index and tab.loc[s].min() >= MIN_CELLS_ROW]
    dropped = {s: dict(tab.loc[s]) for s in tab.index if s not in usable}
    print(f"--- {t} ---")
    if dropped:
        print(f"  dropped (under {MIN_CELLS_ROW} in one condition): {dropped}")
    if not usable:
        print("  nothing passes the threshold"); continue

    sub = sub[state.isin(usable).to_numpy()].copy()
    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})
    n = pd.crosstab(state, cond)

    order = [f"{s}  {c}" for s in usable for c in ["Ctrl", "KD"]]
    labels = {f"{s}  {c}": f"{s}  {c} (n={int(n.loc[s, c]):,})"
              for s in usable for c in ["Ctrl", "KD"]}
    sub.obs["row"] = pd.Categorical((state + "  " + cond).map(labels),
                                    categories=[labels[o] for o in order], ordered=True)

    marks = _resolve(MESO_MARKERS)
    n_genes = sum(len(v) for v in marks.values())

    dp = sc.pl.dotplot(
        sub, marks, groupby="row", categories_order=[labels[o] for o in order],
        use_raw=False, standard_scale="var",
        dot_max=1.0, dot_min=0.0, smallest_dot=14,
        #cmap=LinearSegmentedColormap.from_list("m", [GREY_FAINT, RED]),
        cmap="Reds",
        var_group_rotation=45,
        figsize=(0.30 * n_genes + 2.0, 0.30 * len(order) + 1.4),
        title=None, return_fig=True)
    dp.style(dot_edge_color="black", dot_edge_lw=0.25, largest_dot=190)
    dp.legend(width=1.6, colorbar_title="Scaled expression",
              size_title="Fraction of cells (%)")
    dp.make_figure()
    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels([g.get_text().split(" (")[0] for g in mp.get_xticklabels()],
                           rotation=70, ha="right")
    except Exception as e:
        print(f"  (x-label cleanup skipped: {e})")
    for ext in ["svg", "pdf"]:
        dp.fig.savefig(FIG / f"KD_lnasmelasmmesoderm_dotplot_{t}.{ext}", bbox_inches="tight")
    plt.show()
_sym2key = (dict(zip(adata.var["symbol"].astype(str), adata.var.index))
            if "symbol" in adata.var else {g: g for g in adata.var_names})


In [ ]:
# ---------------------------------------------------------------------------
# Collaborator-style dot plot — KD version. Rows are state x condition so
# Ctrl and KD sit adjacent. Geometry constants at the top, same as Fig 2.
# ---------------------------------------------------------------------------
import re
GENE_W       = 0.4
ROW_H        = 0.30
SMALLEST_DOT = 14
LARGEST_DOT  = 190
DOT_MAX      = 1.0
LEGEND_W     = 1.6
XLAB_ROT     = 70

KD_MATCH_STAN = True   # False = all your cells (primary). True = Stan's gated subset (comparison).
KD_MIN_ROW    = 40      # min cells per state per condition to draw the pair

COLLAB_MARKERS = {
    "PM":         ["MSGN1", "EVX1", "RSPO3", "TBX6", "FOXC1"],
    "somitomere": ["MESP1", "MESP2", "RIPPLY2"],
    "somite":     ["FOXC2", "MEOX1"],
    "PPS":        ["WNT8A", "TBXT"],
    "APS":        ["EOMES", "MIXL1"],
    "DE":         ["FOXA2", "HNF1B"],
    "PGCLC":      ["SOX17", "PRDM1", "NANOS3", "TFAP2C"],
    "pluri":      ["NANOG", "SOX2", "USP44"],
    "AE-LC":      ["GABRP", "WNT6", "ISL1"],
    "apelin":     ["APLNR", "APELA"],
}
COLLAB_ROW_ORDER = ["eNasM", "lNasM", "aPM-LC", "LPM-LC", "CS-LC",
                    "pPS-LC", "aPS-LC", "DE", "PGC-LC", "pluripotent", "AE-LC"]
COLLAB_TIMES = ["42h", "72h"]

_sym2key = (dict(zip(adata.var["symbol"].astype(str), adata.var.index))
            if "symbol" in adata.var else {g: g for g in adata.var_names})
def _resolve(mapping):
    out = {}
    for grp, genes in mapping.items():
        keys = [_sym2key[g] for g in genes if g in _sym2key]
        if keys: out[grp] = keys
    return out

if KD_MATCH_STAN:
    _stan = pd.read_csv(ROOT / "00_DATA_FROZEN" / "01_source" / "sample_barcodes.csv", index_col=0)
    _bc = lambda x: (re.findall(r"[ACGT]{14,}", str(x)) or [None])[0]
    STAN_BC = set(_bc(x) for x in _stan.index)
    adata.obs["_bc"] = [_bc(x) for x in adata.obs_names]

for t in COLLAB_TIMES:
    sub = adata[adata.obs["timepoint_true"].astype(str) == t].copy()
    n_before = sub.n_obs
    if KD_MATCH_STAN:
        sub = sub[sub.obs["_bc"].isin(STAN_BC)].copy()
    print(f"{t}: {n_before:,} -> {sub.n_obs:,} cells"
          + ("  (Stan-gated)" if KD_MATCH_STAN else "  (all cells)"))

    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})
    tab = pd.crosstab(state, cond)
    usable = [s for s in COLLAB_ROW_ORDER
              if s in tab.index and {"Ctrl","KD"}.issubset(tab.columns)
              and tab.loc[s].min() >= KD_MIN_ROW]
    dropped = {s: dict(tab.loc[s]) for s in tab.index if s not in usable}
    if dropped: print(f"  dropped (under {KD_MIN_ROW} in a condition): {dropped}")
    if not usable: print("  nothing passes"); continue

    sub = sub[state.isin(usable).to_numpy()].copy()
    state = sub.obs["predicted_leiden_sub_joint"].astype(str)
    cond  = sub.obs["condition"].astype(str).map({"Ctrl": "Ctrl", "DOX": "KD"})
    n = pd.crosstab(state, cond)
    total = {c: int(n[c].sum()) for c in ["Ctrl", "KD"]}

    order  = [f"{s}  {c}" for s in usable for c in ["Ctrl", "KD"]]
    labels = {f"{s}  {c}": f"{s} {c} (n={int(n.loc[s,c])},p={n.loc[s,c]/total[c]*100:.1f}%)"
              for s in usable for c in ["Ctrl", "KD"]}
    sub.obs["row"] = pd.Categorical((state + "  " + cond).map(labels),
                                    categories=[labels[o] for o in order], ordered=True)

    marks = _resolve(COLLAB_MARKERS)
    n_genes = sum(len(v) for v in marks.values())

    dp = sc.pl.dotplot(
        sub, marks, groupby="row", categories_order=[labels[o] for o in order],
        use_raw=False, standard_scale="var",
        dot_max=DOT_MAX, dot_min=0.0, smallest_dot=SMALLEST_DOT,
        cmap="Reds", var_group_rotation=0,
        figsize=(GENE_W * n_genes, ROW_H * len(order)),
        title=(f"LiAng Amanda {t} Ctrl vs KD" if SHOW_TITLES else None),
        return_fig=True)
    dp.style(dot_edge_color="black", dot_edge_lw=0.25, largest_dot=LARGEST_DOT)
    dp.legend(width=LEGEND_W, colorbar_title="Mean expression\nin group",
              size_title="Fraction of cells\nin group (%)")
    dp.make_figure()
    thin_size_legend(dp, keep=3)
    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels([g.get_text().split(" (")[0] for g in mp.get_xticklabels()],
                           rotation=XLAB_ROT, ha="right")
    except Exception as e:
        print(f"  (x-label cleanup skipped: {e})")
    tag = "STANMATCHED" if KD_MATCH_STAN else "allcells"
    for ext in SAVE_FORMATS:
        dp.fig.savefig(FIG / f"KD_collab_{tag}_{t}.{ext}", bbox_inches="tight")
    plt.show()

## B2 — Ctrl vs KD UMAP (balanced)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
# --- Ctrl vs KD UMAP, both timepoints; KD capped to control N per timepoint ---
# (control shown fully -> matches Fig 2; KD matched down; display only, all cells used for stats)
UMAP_KEY = "X_umap_joint" if "X_umap_joint" in adata.obsm else "X_umap"
xy_all = np.asarray(adata.obsm[UMAP_KEY])
pal = TISSUE_COLORS if "TISSUE_COLORS" in dir() else \
      dict(zip(adata.obs["tissue"].cat.categories,
               plt.get_cmap("tab10")(np.linspace(0,1,adata.obs["tissue"].cat.nunique()))))
rng = np.random.default_rng(0)
# cap = control cell count at each timepoint
ctrl_n = {tp: int(((adata.obs["timepoint_true"].astype(str)==tp) &
                   (adata.obs["cond"].astype(str)=="Ctrl")).sum()) for tp in ["42h","72h"]}
# faint, subsampled backdrop (identical in every panel)
bg = rng.choice(xy_all.shape[0], min(5000, xy_all.shape[0]), replace=False)
panels = [("42h","Ctrl"),("42h","KD"),("72h","Ctrl"),("72h","KD")]
fig, axes = plt.subplots(2, 2, figsize=(11, 10.5))
for ax,(tp,cond) in zip(axes.ravel(), panels):
    m = ((adata.obs["timepoint_true"].astype(str)==tp) & (adata.obs["cond"].astype(str)==cond)).to_numpy()
    idx = np.where(m)[0]
    cap = ctrl_n[tp]
    if len(idx) > cap: idx = rng.choice(idx, cap, replace=False)
    rng.shuffle(idx)
    ax.scatter(xy_all[bg,0], xy_all[bg,1], s=3, c="#ECEAE6", alpha=0.35,
               linewidths=0, rasterized=True, zorder=0)                      # faint backdrop
    tis = adata.obs["tissue"].astype(str).to_numpy()[idx]
    ax.scatter(xy_all[idx,0], xy_all[idx,1], s=20, c=[pal[t] for t in tis],
               linewidths=0, alpha=0.85, rasterized=True, zorder=1)
    ax.set_title(f"{cond}  {tp}   (n={m.sum():,}, shown {len(idx):,})", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal")
handles=[plt.Line2D([0],[0],marker="o",ls="",markersize=30,markerfacecolor=pal[t],
                    markeredgecolor="none",label=t) for t in pal]
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0,0.5), frameon=False)
fig.suptitle("Tissue composition on the UMAP, Ctrl vs KD (KD capped to control N / timepoint)", y=1.0, fontsize=13)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"KD_umap_ctrl_vs_kd_balanced.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# PAPER SOURCE OF TRUTH: time-resolved KD pseudobulk
# Used for state/time KD-vs-Ctrl heatmaps and progression analyses
# ------------------------------------------------------------------
PB_DIR = ROOT / "pseudobulk_ALL_2026-08-13"
PB_FILE = PB_DIR / "ALL_GROUPS_combined.csv"

PB_OUT = TAB / "pseudobulk_KD"
PB_OUT.mkdir(parents=True, exist_ok=True)

if not PB_FILE.exists():
    raise FileNotFoundError(f"Pseudobulk file not found:\n{PB_FILE}")

pb_de = pd.read_csv(PB_FILE)

PB_GROUPS = [
    "pPS-LC_42h",
    "aPS-LC_42h",
    "pPS-LC_72h",
    "aPS-LC_72h",
    "eNasM_72h",
    "lNasM_72h",
]

required_cols = {
    "gene",
    "group",
    "log2FoldChange",
    "log2FC_repeat1_Chicago",
    "log2FC_repeat2_Michigan",
    "repeats_agree",
    "reproduced",
}

missing_cols = required_cols - set(pb_de.columns)
if missing_cols:
    raise ValueError(
        f"Missing pseudobulk columns: {sorted(missing_cols)}"
    )

missing_groups = set(PB_GROUPS) - set(pb_de["group"].unique())
if missing_groups:
    raise ValueError(
        f"Missing pseudobulk groups: {sorted(missing_groups)}"
    )

print("PAPER PSEUDOBULK SOURCE:")
print(PB_FILE)
print(f"Rows: {len(pb_de):,} | genes: {pb_de['gene'].nunique():,}")

print("\nGroups used for progression analyses:")
print(
    pb_de[pb_de["group"].isin(PB_GROUPS)]
    .groupby("group", observed=True)["gene"]
    .nunique()
    .reindex(PB_GROUPS)
    .to_string()
)

# ------------------------------------------------------------------
# New canonical rerun
# DO NOT load unless a downstream analysis specifically requires it.
# Some states are modeled jointly across timepoints, so this is NOT
# interchangeable with pb_de.
# ------------------------------------------------------------------
PB_RERUN_FILE = (
    ROOT
    / "pseudobulk_ALL_2026-08-13"
    / "rerun"
    / "KD_pseudobulk_results.csv"
)

print("\nCanonical rerun available:", PB_RERUN_FILE.exists())

In [ ]:
# ------------------------------------------------------------------
# Main KD pseudobulk heatmap
# Source: pb_de = frozen 08/13 time-resolved DESeq2 pseudobulk
# ------------------------------------------------------------------
from matplotlib.colors import TwoSlopeNorm

GROUPS = [
    "pPS-LC_42h",
    "aPS-LC_42h",
    "pPS-LC_72h",
    "aPS-LC_72h",
    "eNasM_72h",
    "lNasM_72h",
]

XLABS = [
    "pPS 42 h",
    "aPS 42 h",
    "pPS 72 h",
    "aPS 72 h",
    "eNasM 72 h",
    "lNasM 72 h",
]

# Final manual row order
PROGRAMS = {
    "BMP response": [
        "ID3",
        "MSX2",
        "ID1",
    ],

    "Early mesoderm": [
        "MESP1",
        "LHX1",
        "MIXL1",
        "TBXT",
    ],

    "WNT / late patterning": [
        "FGF17",
        "EVX1",
        "RSPO3",
        "DLL3",
        "TBX6",
        "FZD2",
        "WNT8A",
        "SP5",
        "MESP2",
        "FOXC1",
    ],

    "Morphogenesis": [
        "GJA1",
        "VIM",
        "SERPINE2",
        "SNAI1",
        "SPARC",
        "CITED2",
    ],
}

# ------------------------------------------------------------------
# Build pooled and repeat-specific matrices
# ------------------------------------------------------------------
sub = pb_de[
    pb_de["group"].isin(GROUPS)
].copy()

fc = (
    sub.pivot_table(
        index="gene",
        columns="group",
        values="log2FoldChange",
        aggfunc="first",
    )
    .reindex(columns=GROUPS)
)

chi = (
    sub.pivot_table(
        index="gene",
        columns="group",
        values="log2FC_repeat1_Chicago",
        aggfunc="first",
    )
    .reindex(columns=GROUPS)
)

mich = (
    sub.pivot_table(
        index="gene",
        columns="group",
        values="log2FC_repeat2_Michigan",
        aggfunc="first",
    )
    .reindex(columns=GROUPS)
)

# ------------------------------------------------------------------
# Gray only meaningful repeat conflicts.
#
# A tiny opposite-sign effect near zero is treated as neutral:
#   +0.75 / -0.07 -> shown
#
# A real opposite effect is grayed:
#   +0.60 / -0.50 -> gray
# ------------------------------------------------------------------
DISCORDANCE_FLOOR = 0.10

fc_show = fc.copy()

for gene in fc_show.index:
    for group in GROUPS:

        c = chi.loc[gene, group]
        m = mich.loc[gene, group]

        # unavailable in one repeat
        if pd.isna(c) or pd.isna(m):
            fc_show.loc[gene, group] = np.nan
            continue

        meaningful_conflict = (
            np.sign(c) != np.sign(m)
            and abs(c) >= DISCORDANCE_FLOOR
            and abs(m) >= DISCORDANCE_FLOOR
        )

        if meaningful_conflict:
            fc_show.loc[gene, group] = np.nan


# ------------------------------------------------------------------
# Final gene order + program boundaries
# ------------------------------------------------------------------
GENE_ORDER = []
PROGRAM_BOUNDS = []

start = 0

for program, genes in PROGRAMS.items():

    genes = [
        g for g in genes
        if g in fc_show.index
    ]

    if not genes:
        continue

    GENE_ORDER.extend(genes)

    end = start + len(genes) - 1

    PROGRAM_BOUNDS.append(
        (program, start, end)
    )

    start = end + 1


mat = (
    fc_show
    .reindex(
        index=GENE_ORDER,
        columns=GROUPS,
    )
)

# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------
arr = mat.to_numpy(dtype=float)

cmap = plt.cm.RdBu_r.copy()

# Gray = meaningful repeat discordance / unavailable
cmap.set_bad("#D9D9D9")

VMAX = 1.5

norm = TwoSlopeNorm(
    vmin=-VMAX,
    vcenter=0,
    vmax=VMAX,
)

# aspect="equal" below keeps cells square
fig, ax = plt.subplots(
    figsize=(7.8, 9.3)
)

im = ax.imshow(
    np.ma.masked_invalid(arr),
    cmap=cmap,
    norm=norm,
    aspect="equal",
    interpolation="none",
)

# ------------------------------------------------------------------
# Axes
# ------------------------------------------------------------------
ax.set_xticks(
    np.arange(len(GROUPS))
)

ax.set_xticklabels(
    XLABS,
    rotation=45,
    ha="right",
    rotation_mode="anchor",
    fontsize=16,
)

ax.set_yticks(
    np.arange(len(GENE_ORDER))
)

ax.set_yticklabels(
    GENE_ORDER,
    fontsize=16,
)

ax.tick_params(
    axis="both",
    which="both",
    length=0,
)

# ------------------------------------------------------------------
# Thin white borders around every square
# ------------------------------------------------------------------
ax.set_xticks(
    np.arange(-0.5, len(GROUPS), 1),
    minor=True,
)

ax.set_yticks(
    np.arange(-0.5, len(GENE_ORDER), 1),
    minor=True,
)

ax.grid(
    which="minor",
    color="white",
    linewidth=1.1,
)

ax.grid(visible=False)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False,
)

# ------------------------------------------------------------------
# Slightly thicker WHITE vertical breaks
#
# after aPS 42 h:
# pPS42 | aPS42 || pPS72 ...
#
# after aPS 72 h:
# ... pPS72 | aPS72 || eNasM72 ...
# ------------------------------------------------------------------
for x in [
    1.5,
    3.5,
]:
    ax.axvline(
        x,
        color="white",
        linewidth=2.6,
        zorder=5,
    )

# ------------------------------------------------------------------
# Matching horizontal breaks between biological programs
# ------------------------------------------------------------------
for _, _, end in PROGRAM_BOUNDS[:-1]:

    ax.axhline(
        end + 0.5,
        color="white",
        linewidth=2.6,
        zorder=5,
    )

# ------------------------------------------------------------------
# Vertical program labels on left
# ------------------------------------------------------------------
# for program, start, end in PROGRAM_BOUNDS:

#     y_mid = (
#         start + end
#     ) / 2

#     ax.text(
#         -1.35,
#         y_mid,
#         program,
#         rotation=90,
#         ha="center",
#         va="center",
#         fontsize=11.5,
#         fontweight="bold",
#         clip_on=False,
#     )

# ------------------------------------------------------------------
# Clean frame
# ------------------------------------------------------------------
for spine in ax.spines.values():
    spine.set_visible(False)

cb = fig.colorbar(
    im,
    ax=ax,
    fraction=0.040,
    pad=0.035,
)

cb.set_label(
    "KD vs Ctrl log$_2$FC",
    fontsize=11,
)

cb.ax.tick_params(
    labelsize=10
)

fig.tight_layout()

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
stem = "KD_pseudobulk_mesoderm_progression_heatmap"

for ext in ["svg", "pdf"]:
    fig.savefig(
        FIG / f"{stem}.{ext}",
        bbox_inches="tight",
        facecolor="white",
    )

# Save exact plotted values for audit
mat.to_csv(
    PB_OUT / f"{stem}_values.csv"
)

plt.show()

print("Saved:")
print(FIG / f"{stem}.svg")
print(FIG / f"{stem}.pdf")


In [ ]:
# ============================================================================
# F — LOAD CTRL-DEFINED APLNR WGCNA MODULE
# Source: Figure 2 Ctrl mesoderm-only WGCNA
# Score in KD after EXCLUDING APLNR itself
# ============================================================================

WGCNA_TAB = ROOT / "06_tables" / "FIGURE2_clean"

MODULE_FILE = (
    WGCNA_TAB
    / "APLNR_Ctrl_mesoOnly_WGCNA_black_module_with_kME.csv"
)

MODULE_OUT = TAB / "APLNR_module_KD"
MODULE_OUT.mkdir(parents=True, exist_ok=True)

if not MODULE_FILE.exists():
    raise FileNotFoundError(MODULE_FILE)

module_def = pd.read_csv(MODULE_FILE)

print("Module file:", MODULE_FILE)
print("Columns:", module_def.columns.tolist())
print("Rows:", len(module_def))

# Frozen Ctrl-defined APLNR-containing WGCNA module
black_genes_all = (
    module_def["gene"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

# Exclude the perturbed gene itself from downstream score
black_genes_score = [
    g for g in black_genes_all
    if g != "APLNR"
]

print("\nCtrl-defined module:", len(black_genes_all), "genes")
print("APLNR present:", "APLNR" in black_genes_all)
print("Genes used for KD score:", len(black_genes_score))
print("APLNR used in score:", "APLNR" in black_genes_score)

In [ ]:
# ============================================================================
# F — SCORE THE CTRL-DEFINED APLNR WGCNA MODULE IN CTRL vs APLNR KD
#
# WGCNA itself is NOT rerun.
# The frozen Ctrl-defined 218-gene module is loaded from Figure 2.
# APLNR itself is excluded from the downstream score -> 217-gene score.
#
# Requires upstream:
#   ROOT, MASTER, STAN_CSV, TAB
# ============================================================================

import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad


# ============================================================================
# 1. LOAD THE FROZEN CTRL-DEFINED WGCNA MODULE
# ============================================================================

WGCNA_TAB = (
    ROOT
    / "06_tables"
    / "FIGURE2_clean"
)

MODULE_FILE = (
    WGCNA_TAB
    / "APLNR_Ctrl_mesoOnly_WGCNA_black_module_with_kME.csv"
)

MODULE_OUT = (
    TAB
    / "APLNR_module_KD"
)

MODULE_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

if not MODULE_FILE.exists():
    raise FileNotFoundError(MODULE_FILE)

module_def = pd.read_csv(
    MODULE_FILE
)

black_genes_all = (
    module_def["gene"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

# IMPORTANT:
# keep the WGCNA module definition unchanged,
# but exclude the experimentally perturbed gene from the score itself.
black_genes_score = [
    g
    for g in black_genes_all
    if g != "APLNR"
]

print("WGCNA module file:")
print(MODULE_FILE)

print(
    f"\nFrozen Ctrl-defined module: "
    f"{len(black_genes_all)} genes"
)

print(
    "APLNR present in original module:",
    "APLNR" in black_genes_all,
)

print(
    f"Genes used for KD score: "
    f"{len(black_genes_score)}"
)

print(
    "APLNR included in score:",
    "APLNR" in black_genes_score,
)


# ============================================================================
# 2. LOAD A FRESH SINGLE-CELL OBJECT
#
# Do NOT use the already-filtered `adata` object here.
# This reproduces the filtering used in the original Figure 2 score.
# ============================================================================

adata_score = ad.read_h5ad(
    MASTER
)


# ============================================================================
# 3. STAN BARCODE FILTER
# Ctrl gets the Stan filter; KD is retained normally.
# ============================================================================

def bc16(x):
    h = re.findall(
        r"[ACGT]{14,}",
        str(x),
    )
    return h[0] if h else None


stan = pd.read_csv(
    STAN_CSV,
    index_col=0,
)

STAN_BC_SCORE = {
    bc16(x)
    for x in stan.index
    if bc16(x) is not None
}

adata_score.obs["_bc_score"] = [
    bc16(x)
    for x in adata_score.obs_names
]


STATE_COL = "predicted_leiden_sub_joint"
KEEP_TIMES = ["42h", "72h"]


# ============================================================================
# 4. IDENTIFY KD CONDITION
# ============================================================================

conditions = (
    adata_score.obs["condition"]
    .astype(str)
    .unique()
    .tolist()
)

if "DOX" in conditions:

    KD_LABEL = "DOX"

else:

    other = [
        x
        for x in conditions
        if x != "Ctrl"
    ]

    assert len(other) == 1, (
        "More than one non-Ctrl condition found. "
        "Set KD_LABEL manually."
    )

    KD_LABEL = other[0]


print(
    "\nKD condition:",
    KD_LABEL,
)


# ============================================================================
# 5. FILTER CELLS
# ============================================================================

cond = (
    adata_score.obs["condition"]
    .astype(str)
)

keep_condition = cond.isin(
    ["Ctrl", KD_LABEL]
)

keep_time = (
    adata_score.obs["timepoint_true"]
    .astype(str)
    .isin(KEEP_TIMES)
)

# exactly as in original score:
# Ctrl must match Stan barcodes;
# KD does not receive the Stan restriction.
keep_stan = (
    (cond != "Ctrl")
    | adata_score.obs["_bc_score"].isin(
        STAN_BC_SCORE
    )
)

cmp = adata_score[
    keep_condition
    & keep_time
    & keep_stan
].copy()


# Original state annotation
cmp.obs["state"] = (
    cmp.obs[STATE_COL]
    .astype(str)
)


print(
    f"\nCells used: "
    f"{cmp.n_obs:,}"
)

print(
    cmp.obs.groupby(
        [
            "condition",
            "site",
            "timepoint_true",
            "state",
        ],
        observed=True,
    )
    .size()
    .to_string()
)


# ============================================================================
# 6. RAW COUNTS
# ============================================================================

assert "counts" in cmp.layers

X = cmp.layers["counts"]

if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()


# whole-cell library size
library_size = np.asarray(
    X.sum(axis=1)
).ravel()


# ============================================================================
# 7. MAP GENE SYMBOLS TO RAW-COUNT COLUMNS
#
# If a gene symbol occurs more than once,
# sum all corresponding raw-count columns.
# ============================================================================

if "symbol" in cmp.var.columns:

    symbols = (
        cmp.var["symbol"]
        .astype(str)
        .values
    )

else:

    symbols = (
        cmp.var_names
        .astype(str)
        .values
    )


symbol_to_cols = {}

for j, g in enumerate(symbols):

    symbol_to_cols.setdefault(
        g,
        [],
    ).append(j)


genes_present = [
    g
    for g in black_genes_score
    if g in symbol_to_cols
]

genes_missing = [
    g
    for g in black_genes_score
    if g not in symbol_to_cols
]


print(
    f"\nModule genes found: "
    f"{len(genes_present)}/"
    f"{len(black_genes_score)}"
)

if genes_missing:
    print(
        "Missing:",
        genes_missing,
    )


# ============================================================================
# 8. BUILD CELL x MODULE-GENE RAW COUNT MATRIX
# ============================================================================

module_columns = []

for g in genes_present:

    cols = symbol_to_cols[g]

    if len(cols) == 1:

        v = X[:, cols[0]]

    else:

        v = X[:, cols].sum(
            axis=1
        )

    module_columns.append(
        sp.csr_matrix(v)
    )


Xmod = sp.hstack(
    module_columns,
    format="csr",
)


# ============================================================================
# 9. WHOLE-CELL CP10K -> log1p
# ============================================================================

scale = np.divide(
    1e4,
    library_size,
    out=np.zeros_like(
        library_size,
        dtype=float,
    ),
    where=library_size > 0,
)

Xmod = (
    sp.diags(scale)
    @ Xmod
)

Xmod = (
    Xmod
    .toarray()
    .astype(np.float32)
)

Xmod = np.log1p(
    Xmod
)


# ============================================================================
# 10. CTRL-REFERENCED MODULE SCORE
#
# For each module gene:
#
#   z = (cell expression - Ctrl mean) / Ctrl SD
#
# Module score:
#
#   mean z across all usable module genes
#
# APLNR ITSELF IS NOT INCLUDED.
# ============================================================================

ctrl_mask = (
    cmp.obs["condition"]
    .astype(str)
    .values
    == "Ctrl"
)

ctrl_mu = (
    Xmod[ctrl_mask]
    .mean(axis=0)
)

ctrl_sd = (
    Xmod[ctrl_mask]
    .std(axis=0)
)

# Remove genes with no variance in Ctrl
usable = (
    ctrl_sd
    > 1e-6
)


print(
    f"\nGenes contributing to score: "
    f"{usable.sum()}/"
    f"{len(genes_present)}"
)


Z = (
    Xmod[:, usable]
    - ctrl_mu[usable]
) / ctrl_sd[usable]


cmp.obs[
    "APLNR_module_score_noAPLNR"
] = (
    Z.mean(axis=1)
)

# ============================================================================
# DECOMPOSE MODULE SCORE INTO PER-GENE CONTRIBUTIONS
#
# Each column of Z is the Ctrl-standardized expression of one module gene.
# Therefore:
#
#   module score = mean(z_gene1, z_gene2, ...)
#
# and:
#
#   KD - Ctrl module score
#       = mean(KD - Ctrl difference for each gene)
#
# ============================================================================

# Names of genes that actually contributed to the score
score_genes = np.array(genes_present)[usable]

assert len(score_genes) == Z.shape[1]

# Put gene-level z-scores into a dataframe
Z_df = pd.DataFrame(
    Z,
    index=cmp.obs_names,
    columns=score_genes,
)

# Metadata needed for grouping
meta = cmp.obs[
    [
        "site",
        "timepoint_true",
        "state",
        "condition",
    ]
].copy()

# merge eNasM + lNasM for the main analysis
meta["state_plot"] = (
    meta["state"]
    .astype(str)
    .replace({
        "eNasM": "NasM",
        "lNasM": "NasM",
    })
)

# attach metadata to gene-level z-scores
gene_score_df = pd.concat(
    [
        meta,
        Z_df,
    ],
    axis=1,
)

print(
    "Genes decomposed:",
    len(score_genes),
)

# ============================================================================
# PER-GENE KD - CTRL EFFECT WITHIN EACH SITE x TIME x STATE
# ============================================================================

long = (
    gene_score_df
    .melt(
        id_vars=[
            "site",
            "timepoint_true",
            "state_plot",
            "condition",
        ],
        value_vars=score_genes,
        var_name="gene",
        value_name="gene_z",
    )
)

# Mean gene z-score within each biological repeat / state / condition
gene_means = (
    long
    .groupby(
        [
            "site",
            "timepoint_true",
            "state_plot",
            "condition",
            "gene",
        ],
        observed=True,
    )["gene_z"]
    .mean()
    .reset_index()
)

# Ctrl vs KD side-by-side
gene_wide = (
    gene_means
    .pivot_table(
        index=[
            "site",
            "timepoint_true",
            "state_plot",
            "gene",
        ],
        columns="condition",
        values="gene_z",
    )
    .reset_index()
)

assert "Ctrl" in gene_wide.columns
assert KD_LABEL in gene_wide.columns

gene_wide["KD_minus_Ctrl"] = (
    gene_wide[KD_LABEL]
    - gene_wide["Ctrl"]
)

display(
    gene_wide.head()
)


# ============================================================================
# SUMMARIZE PER-GENE EFFECT ACROSS CHICAGO + MICHIGAN
# ============================================================================

gene_delta_summary = (
    gene_wide
    .groupby(
        [
            "timepoint_true",
            "state_plot",
            "gene",
        ],
        observed=True,
    )["KD_minus_Ctrl"]
    .agg(
        mean_delta="mean",
        min_delta="min",
        max_delta="max",
        n_sites="count",
    )
    .reset_index()
)

site_sign = (
    gene_wide
    .assign(
        sign=np.sign(
            gene_wide["KD_minus_Ctrl"]
        )
    )
    .groupby(
        [
            "timepoint_true",
            "state_plot",
            "gene",
        ],
        observed=True,
    )["sign"]
    .agg(
        lambda x:
        len(set(x[x != 0])) <= 1
    )
    .rename("sites_same_direction")
    .reset_index()
)

gene_delta_summary = (
    gene_delta_summary
    .merge(
        site_sign,
        on=[
            "timepoint_true",
            "state_plot",
            "gene",
        ],
        how="left",
    )
)

# Truly reproducible = both Chicago + Michigan present
# AND KD effect has the same sign in both.
gene_delta_summary["reproduced"] = (
    (gene_delta_summary["n_sites"] == 2)
    & gene_delta_summary["sites_same_direction"]
)

gene_delta_summary.to_csv(
    MODULE_OUT
    / "APLNR_module_noAPLNR_per_gene_KD_minus_Ctrl.csv",
    index=False,
)

# ============================================================================
# 11. SAVE ORIGINAL eNasM / lNasM STATES SEPARATELY
#
# Keep this for audit and supplementary analyses.
# ============================================================================

site_score_separate = (
    cmp.obs
    .groupby(
        [
            "site",
            "timepoint_true",
            "state",
            "condition",
        ],
        observed=True,
    )[
        "APLNR_module_score_noAPLNR"
    ]
    .mean()
    .reset_index()
)


SEPARATE_FILE = (
    MODULE_OUT
    / "APLNR_module_noAPLNR_score_site_means_separateNasM.csv"
)

site_score_separate.to_csv(
    SEPARATE_FILE,
    index=False,
)


# ============================================================================
# 12. MERGE eNasM + lNasM FOR THE MAIN FIGURE
#
# IMPORTANT:
# merge at the CELL level first, then average.
# We are NOT averaging the eNasM and lNasM means.
# ============================================================================

cmp.obs["state_plot"] = (
    cmp.obs["state"]
    .replace(
        {
            "eNasM": "NasM",
            "lNasM": "NasM",
        }
    )
)


site_score_plot = (
    cmp.obs
    .groupby(
        [
            "site",
            "timepoint_true",
            "state_plot",
            "condition",
        ],
        observed=True,
    )[
        "APLNR_module_score_noAPLNR"
    ]
    .mean()
    .reset_index()
)


MERGED_FILE = (
    MODULE_OUT
    / "APLNR_module_noAPLNR_score_site_means_mergedNasM.csv"
)

site_score_plot.to_csv(
    MERGED_FILE,
    index=False,
)


# ============================================================================
# 13. DISPLAY THE STATES WE CARE ABOUT FOR FIGURE F
# ============================================================================

figure_states = (
    site_score_plot[
        (
            (
                site_score_plot["timepoint_true"].astype(str) == "42h"
            )
            &
            (
                site_score_plot["state_plot"].isin([
                    "pPS-LC",
                    "aPS-LC",
                ])
            )
        )
        |
        (
            (
                site_score_plot["timepoint_true"].astype(str) == "72h"
            )
            &
            (
                site_score_plot["state_plot"].isin([
                    "pPS-LC",
                    "aPS-LC",
                    "NasM",
                    "aPM-LC",
                ])
            )
        )
    ]
    .copy()
)

# ============================================================================
# WHICH MODULE GENES DRIVE THE NasM 72 h KD EFFECT?
# ============================================================================

nasm_gene_effect = (
    gene_delta_summary[
        (gene_delta_summary["timepoint_true"].astype(str) == "72h")
        & (gene_delta_summary["state_plot"].astype(str) == "NasM")
    ]
    .sort_values("mean_delta")
)

display(
    nasm_gene_effect[
        [
            "gene",
            "mean_delta",
            "min_delta",
            "max_delta",
            "n_sites",
            "reproduced",
        ]
    ].head(30)
)

display(
    figure_states.sort_values(
        [
            "timepoint_true",
            "state_plot",
            "site",
            "condition",
        ]
    )
)


print(
    "\nSaved separate NasM table:"
)
print(
    SEPARATE_FILE
)

print(
    "\nSaved merged NasM table:"
)
print(
    MERGED_FILE
)

In [ ]:
# ============================================================================
# G — FUNCTIONAL DECOMPOSITION OF THE NasM MODULE DEFICIT
#
# Question:
# Which functions enriched in the original Ctrl-defined APLNR module
# contain genes that are most consistently reduced in KD NasM?
#
# IMPORTANT:
# - GO enrichment was defined from the Ctrl module.
# - KD effects come from gene_delta_summary.
# - This is a decomposition of the module effect, not a new GO test of KD.
# ============================================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# 1. Load GO enrichment of the original Ctrl-defined APLNR module
# ---------------------------------------------------------------------------

GO_FILE = (
    WGCNA_TAB
    / "APLNR_mesoderm_module_GO_enrichment_ALL.csv"
)

if not GO_FILE.exists():
    raise FileNotFoundError(GO_FILE)

go = pd.read_csv(GO_FILE)


# ---------------------------------------------------------------------------
# 2. Parse module genes belonging to each GO term
# ---------------------------------------------------------------------------

go["gene_list"] = (
    go["Genes"]
    .fillna("")
    .astype(str)
    .apply(
        lambda value: [
            gene.strip()
            for gene in value.split(";")
            if gene.strip()
        ]
    )
)

go["n_module_genes"] = (
    go["gene_list"]
    .apply(len)
)


# Retain significant Ctrl-module GO terms with enough genes
go_use = (
    go[
        (go["Adjusted P-value"] < 0.05)
        & (go["n_module_genes"] >= 4)
    ]
    .copy()
)

print(
    "Significant Ctrl-module GO terms with >=4 genes:",
    len(go_use),
)


# ---------------------------------------------------------------------------
# 3. Select per-gene effects in combined NasM at 72 h
#
# gene_delta_summary was created by the preceding module-decomposition block.
# ---------------------------------------------------------------------------

nasm_gene = (
    gene_delta_summary[
        (
            gene_delta_summary["timepoint_true"]
            .astype(str)
            == "72h"
        )
        &
        (
            gene_delta_summary["state_plot"]
            .astype(str)
            == "NasM"
        )
        &
        (
            gene_delta_summary["n_sites"]
            == 2
        )
    ]
    .copy()
)

print(
    "Module genes measured in both repeats for NasM 72 h:",
    nasm_gene["gene"].nunique(),
)


# ---------------------------------------------------------------------------
# 4. Summarize KD effects within each Ctrl-defined GO category
# ---------------------------------------------------------------------------

rows = []

for _, term_row in go_use.iterrows():

    d = nasm_gene[
        nasm_gene["gene"].isin(
            term_row["gene_list"]
        )
    ].copy()

    if len(d) < 4:
        continue

    rows.append({

        "Term": term_row["Term"],

        # This P value describes enrichment in the original Ctrl module.
        # It is NOT a test of the KD effect.
        "Ctrl_module_GO_padj":
            term_row["Adjusted P-value"],

        "n_genes": len(d),

        # Average KD − Ctrl effect across member genes
        "mean_KD_minus_Ctrl":
            d["mean_delta"].mean(),

        "median_KD_minus_Ctrl":
            d["mean_delta"].median(),

        # Negative based on the mean of the two repeats
        "fraction_down_mean":
            np.mean(d["mean_delta"] < 0),

        # Negative in both repeats:
        # max_delta < 0 means even the less-negative repeat is below zero.
        "fraction_down_both":
            np.mean(d["max_delta"] < 0),

        # Positive in both repeats
        "fraction_up_both":
            np.mean(d["min_delta"] > 0),

        # Same sign in both repeats, whether up or down
        "fraction_reproduced":
            np.mean(d["reproduced"]),

        "genes":
            ";".join(d["gene"].astype(str)),
    })


go_kd = pd.DataFrame(rows)

if go_kd.empty:
    raise ValueError(
        "No GO terms retained after intersecting with NasM genes."
    )

go_kd = (
    go_kd
    .sort_values(
        [
            "mean_KD_minus_Ctrl",
            "fraction_down_both",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 5. Save and display
# ---------------------------------------------------------------------------

GO_KD_FILE = (
    MODULE_OUT
    / "APLNR_module_GO_terms_NasM72_KD_effect.csv"
)

go_kd.to_csv(
    GO_KD_FILE,
    index=False,
)

display(
    go_kd[
        [
            "Term",
            "n_genes",
            "mean_KD_minus_Ctrl",
            "median_KD_minus_Ctrl",
            "fraction_down_mean",
            "fraction_down_both",
            "fraction_reproduced",
            "Ctrl_module_GO_padj",
        ]
    ].head(30)
)

print("\nSaved:")
print(GO_KD_FILE)

In [ ]:
# ============================================================================
# GO BIOLOGICAL PROCESS GSEA
#
# Run after the Hallmark GSEA block.
# Reuses: pb_de, RANK_COL, GSEA_GROUPS, TAB
#
# Positive NES = enriched in APLNR KD
# Negative NES = reduced in APLNR KD
# ============================================================================

GO_LIBRARY = "GO_Biological_Process_2026"


# ------------------------------------------------------------------
# Load GO Biological Process gene sets
# ------------------------------------------------------------------

go_sets = gp.get_library(
    name=GO_LIBRARY,
    organism="Human",
)

print(
    f"{GO_LIBRARY} gene sets loaded:",
    len(go_sets),
)


# ------------------------------------------------------------------
# Run exploratory GO GSEA
#
# GO contains thousands of sets, so use 1,000 permutations initially.
# ------------------------------------------------------------------

go_gsea_results = []
go_gsea_objects = {}

for group in GSEA_GROUPS:

    d = (
        pb_de.loc[
            pb_de["group"].astype(str).eq(group),
            ["gene", RANK_COL],
        ]
        .dropna()
        .copy()
    )

    d[RANK_COL] = pd.to_numeric(
        d[RANK_COL],
        errors="coerce",
    )

    d = d[np.isfinite(d[RANK_COL])]

    # Resolve duplicated gene symbols
    d["abs_rank"] = d[RANK_COL].abs()

    d = (
        d.sort_values("abs_rank", ascending=False)
         .drop_duplicates("gene", keep="first")
    )

    ranking = (
        d.set_index("gene")[RANK_COL]
         .sort_values(ascending=False)
    )

    print(
        f"{group}: {len(ranking):,} ranked genes"
    )

    pre = gp.prerank(
        rnk=ranking,
        gene_sets=go_sets,
        min_size=5,
        max_size=500,
        permutation_num=1_000,
        seed=42,
        threads=4,
        verbose=False,
    )

    result = pre.res2d.copy()
    result["group"] = group

    go_gsea_results.append(result)
    go_gsea_objects[group] = pre


# ------------------------------------------------------------------
# Combine and save complete results
# ------------------------------------------------------------------

go_gsea = pd.concat(
    go_gsea_results,
    ignore_index=True,
)

for col in ["NES", "NOM p-val", "FDR q-val"]:
    if col in go_gsea.columns:
        go_gsea[col] = pd.to_numeric(
            go_gsea[col],
            errors="coerce",
        )

go_gsea.to_csv(
    TAB / "APLNR_KD_GO_BP_2026_preranked_GSEA.csv",
    index=False,
)

print(
    "Total GO GSEA results:",
    len(go_gsea),
)


# ------------------------------------------------------------------
# Extract developmentally relevant terms
# ------------------------------------------------------------------

DEVELOPMENT_PATTERN = (
    r"BMP|NODAL|MESODERM|MESENCHYM|"
    r"GASTRULATION|MIGRATION|MOTILITY|CHEMOTAX|"
    r"EPITHELIAL.TO.MESENCHYMAL|"
    r"HEART|CARDIAC|CARDIOVASCULAR|"
    r"PARAXIAL|SOMIT|LATERAL.PLATE|"
    r"WNT|MORPHOGEN"
)

developmental_gsea = (
    go_gsea[
        go_gsea["Term"]
        .astype(str)
        .str.contains(
            DEVELOPMENT_PATTERN,
            case=False,
            regex=True,
            na=False,
        )
    ]
    .sort_values(
        ["group", "FDR q-val", "NES"],
        ascending=[True, True, True],
    )
)

display(
    developmental_gsea[
        [
            "group",
            "Term",
            "NES",
            "NOM p-val",
            "FDR q-val",
            "Lead_genes",
        ]
    ]
)

developmental_gsea.to_csv(
    TAB / "APLNR_KD_GO_BP_2026_developmental_terms.csv",
    index=False,
)


# ------------------------------------------------------------------
# Significant developmental terms only
# ------------------------------------------------------------------

developmental_sig = (
    developmental_gsea[
        developmental_gsea["FDR q-val"] < 0.05
    ]
    .copy()
)

print(
    "Significant developmental GO results:",
    len(developmental_sig),
)

display(
    developmental_sig[
        [
            "group",
            "Term",
            "NES",
            "FDR q-val",
            "Lead_genes",
        ]
    ]
)

In [ ]:
# ============================================================================
# PRERANKED GSEA FROM THE EXISTING PSEUDOBULK DIFFERENTIAL-EXPRESSION TABLE
#
# Uses:
#   pb_de["gene"]
#   pb_de["group"]
#   preferably a DESeq2 Wald statistic; otherwise log2FoldChange
#
# Positive NES = enriched in APLNR KD
# Negative NES = enriched in Control / reduced in APLNR KD
# ============================================================================

import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt

# If needed:
# %pip install gseapy


# ------------------------------------------------------------------
# 1. Choose the best available ranking statistic
# ------------------------------------------------------------------

RANK_CANDIDATES = [
    "stat",                 # standard DESeq2 Wald statistic
    "wald_stat",
    "Wald",
    "score",
    "log2FoldChange",       # fallback only
]

RANK_COL = next(
    (col for col in RANK_CANDIDATES if col in pb_de.columns),
    None,
)

if RANK_COL is None:
    raise ValueError(
        "No usable ranking column found. Available columns are:\n"
        + "\n".join(pb_de.columns.astype(str))
    )

print("GSEA ranking column:", RANK_COL)

if RANK_COL == "log2FoldChange":
    print(
        "WARNING: No Wald statistic was found, so GSEA will be ranked by "
        "log2FoldChange. This is suitable for exploration, but a DESeq2 "
        "Wald statistic would be preferable for the final analysis."
    )


# ------------------------------------------------------------------
# 2. Download an independently defined Hallmark gene-set collection
# ------------------------------------------------------------------

hallmark_sets = gp.get_library(
    name="MSigDB_Hallmark_2020",
    organism="Human",
)

print("Hallmark gene sets loaded:", len(hallmark_sets))


# ------------------------------------------------------------------
# 3. Contrasts to analyze
#
# Keep eNasM and lNasM separate unless DESeq2 was rerun after genuinely
# merging their cells into a single NasM pseudobulk population.
# ------------------------------------------------------------------

GSEA_GROUPS = [
    "all_mesoderm_42h",
    "all_mesoderm_72h",
    "pPS-LC_42h",
    "aPS-LC_42h",
    "pPS-LC_72h",
    "aPS-LC_72h",
    "NasM_72h",
]


# ------------------------------------------------------------------
# 4. Run preranked GSEA
# ------------------------------------------------------------------

gsea_results = []
gsea_objects = {}

for group in GSEA_GROUPS:

    d = (
        pb_de.loc[
            pb_de["group"].astype(str).eq(group),
            ["gene", RANK_COL],
        ]
        .dropna()
        .copy()
    )

    # Remove non-finite statistics
    d[RANK_COL] = pd.to_numeric(d[RANK_COL], errors="coerce")
    d = d[np.isfinite(d[RANK_COL])]

    # Resolve duplicated gene symbols, if any
    # Retain the value with the largest absolute ranking statistic
    d["abs_rank"] = d[RANK_COL].abs()
    d = (
        d.sort_values("abs_rank", ascending=False)
         .drop_duplicates("gene", keep="first")
    )

    ranking = (
        d.set_index("gene")[RANK_COL]
         .sort_values(ascending=False)
    )

    print(
        f"{group}: {len(ranking):,} ranked genes; "
        f"range = {ranking.min():.3f} to {ranking.max():.3f}"
    )

    pre = gp.prerank(
        rnk=ranking,
        gene_sets=hallmark_sets,
        min_size=10,
        max_size=500,
        permutation_num=10_000,
        seed=42,
        threads=4,
        verbose=False,
    )

    result = pre.res2d.copy()
    result["group"] = group

    gsea_results.append(result)
    gsea_objects[group] = pre


gsea = pd.concat(
    gsea_results,
    ignore_index=True,
)

# Standardize numeric columns
for col in ["NES", "NOM p-val", "FDR q-val"]:
    if col in gsea.columns:
        gsea[col] = pd.to_numeric(gsea[col], errors="coerce")

gsea.to_csv(
    TAB / "APLNR_KD_Hallmark_preranked_GSEA.csv",
    index=False,
)

display(
    gsea.sort_values(
        ["group", "FDR q-val", "NES"],
        ascending=[True, True, True],
    )
)


# ------------------------------------------------------------------
# 5. Inspect pathways most relevant to the paper
# ------------------------------------------------------------------

FOCUS_PATTERN = (
    "WNT|BETA.CATENIN|EPITHELIAL.MESENCHYMAL|"
    "TGF.BETA|APICAL.JUNCTION|MYOGENESIS"
)

gsea_focus = (
    gsea[
        gsea["Term"]
        .astype(str)
        .str.contains(
            FOCUS_PATTERN,
            case=False,
            regex=True,
            na=False,
        )
    ]
    .sort_values(
        ["group", "FDR q-val"],
        ascending=[True, True],
    )
)

display(
    gsea_focus[
        [
            "group",
            "Term",
            "NES",
            "NOM p-val",
            "FDR q-val",
            "Lead_genes",
        ]
    ]
)

gsea_focus.to_csv(
    TAB / "APLNR_KD_Hallmark_GSEA_focused_pathways.csv",
    index=False,
)

In [ ]:
gsea_sig = (
    gsea.loc[
        gsea["FDR q-val"] < 0.05,
        [
            "group",
            "Term",
            "NES",
            "NOM p-val",
            "FDR q-val",
            "Lead_genes",
        ],
    ]
    .sort_values(
        ["group", "NES"],
        ascending=[True, False],
    )
)

display(gsea_sig)

gsea_sig.to_csv(
    TAB / "APLNR_KD_Hallmark_GSEA_significant.csv",
    index=False,
)

In [ ]:
# ============================================================================
# F — EFFECT OF APLNR KD ON THE CTRL-DEFINED APLNR-ASSOCIATED MODULE
#
# Module:
#   Ctrl-defined WGCNA module
#   APLNR itself excluded from the score
#
# Plot:
#   KD - Ctrl calculated independently within Chicago and Michigan
#   Open points = biological repeats
#   Filled point = mean across repeats
# ============================================================================

# ------------------------------------------------------------------
# States / timepoints to show
# ------------------------------------------------------------------

PLOT_STATES = [
    ("42h", "pPS-LC", "pPS"),
    ("42h", "aPS-LC", "aPS"),
    ("72h", "pPS-LC", "pPS"),
    ("72h", "aPS-LC", "aPS"),
    ("72h", "NasM",   "NasM"),
    ("72h", "aPM-LC", "aPM"),
]

# gap between 42 h and 72 h groups
XPOS = [0, 1, 3, 4, 5, 6]

parts = []

for x, (tp, state, label) in zip(XPOS, PLOT_STATES):

    d = site_score_plot[
        (site_score_plot["timepoint_true"].astype(str) == tp)
        &
        (site_score_plot["state_plot"].astype(str) == state)
    ].copy()

    d["x"] = x
    d["label"] = label

    parts.append(d)

plot_score = pd.concat(
    parts,
    ignore_index=True,
)


# ============================================================================
# CALCULATE KD - CTRL WITHIN EACH BIOLOGICAL REPEAT
# ============================================================================

wide = (
    plot_score
    .pivot_table(
        index=[
            "site",
            "timepoint_true",
            "state_plot",
            "x",
            "label",
        ],
        columns="condition",
        values="APLNR_module_score_noAPLNR",
    )
    .reset_index()
)

assert "Ctrl" in wide.columns
assert KD_LABEL in wide.columns

wide["KD_minus_Ctrl"] = (
    wide[KD_LABEL]
    - wide["Ctrl"]
)


# ------------------------------------------------------------------
# Mean across Chicago + Michigan
# ------------------------------------------------------------------

mean_delta = (
    wide
    .groupby(
        [
            "timepoint_true",
            "state_plot",
            "x",
            "label",
        ],
        observed=True,
    )["KD_minus_Ctrl"]
    .agg(
        mean_delta="mean",
        n_sites="count",
    )
    .reset_index()
)


display(
    wide[
        [
            "site",
            "timepoint_true",
            "state_plot",
            "KD_minus_Ctrl",
        ]
    ]
)

display(mean_delta)


# ============================================================================
# PLOT
# ============================================================================

FIG_C = "#D27D72"

fig, ax = plt.subplots(
    figsize=(5.4, 3.8)
)


# ------------------------------------------------------------------
# Individual biological repeats
# ------------------------------------------------------------------

for _, row in wide.iterrows():

    ax.scatter(
        row["x"],
        row["KD_minus_Ctrl"],
        s=42,
        facecolor="white",
        edgecolor=FIG_C,
        linewidth=1.4,
        zorder=3,
    )


# ------------------------------------------------------------------
# Mean across Chicago + Michigan
# ------------------------------------------------------------------

ax.scatter(
    mean_delta["x"],
    mean_delta["mean_delta"],
    s=72,
    color=FIG_C,
    edgecolor="white",
    linewidth=0.8,
    zorder=4,
)


# ------------------------------------------------------------------
# Optional: tiny vertical line showing range of the two repeats
# ------------------------------------------------------------------

for _, row in mean_delta.iterrows():

    vals = wide[
        wide["x"] == row["x"]
    ]["KD_minus_Ctrl"].dropna()

    if len(vals) >= 2:

        ax.plot(
            [row["x"], row["x"]],
            [vals.min(), vals.max()],
            color=FIG_C,
            linewidth=1.0,
            alpha=0.45,
            zorder=1,
        )


# ------------------------------------------------------------------
# Zero = no KD effect
# ------------------------------------------------------------------

ax.axhline(
    0,
    color="#9E9E9E",
    linewidth=0.9,
    zorder=0,
)


# ------------------------------------------------------------------
# x axis
# ------------------------------------------------------------------

ax.set_xticks(XPOS)

ax.set_xticklabels(
    [x[2] for x in PLOT_STATES],
    fontsize=10,
)


# Timepoint labels above groups
ymin, ymax = ax.get_ylim()

ax.text(
    0.5,
    ymax,
    "42 h",
    ha="center",
    va="bottom",
    fontsize=10,
)

ax.text(
    4.5,
    ymax,
    "72 h",
    ha="center",
    va="bottom",
    fontsize=10,
)


# subtle separator between timepoints
ax.axvline(
    2,
    color="#D9D9D9",
    linewidth=0.8,
    zorder=0,
)


# ------------------------------------------------------------------
# Labels / styling
# ------------------------------------------------------------------

ax.set_ylabel(
    "KD − Ctrl module score",
    fontsize=11,
)

ax.set_title(
    "Ctrl-defined APLNR-associated program",
    fontsize=12,
)

ax.grid(False)

ax.tick_params(
    axis="both",
    length=0,
)

for spine in [
    "top",
    "right",
]:
    ax.spines[spine].set_visible(False)


fig.tight_layout()


# ============================================================================
# SAVE
# ============================================================================

for ext in SAVE_FORMATS:

    fig.savefig(
        FIG
        / f"KD_APLNR_module_effect_by_state_noAPLNR.{ext}",
        bbox_inches="tight",
    )


plt.show()

In [ ]:
# ============================================================================
# MAIN FIGURE — MEAN KD EFFECT + RANGE ACROSS BIOLOGICAL REPLICATES
#
# Point    = mean KD − Ctrl module-score difference
# Whiskers = range of the two biological replicates
#
# Requires:
#   wide, FIG, SAVE_FORMATS
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# States and positions
# ---------------------------------------------------------------------------

PLOT_GROUPS = [
    ("42h", "pPS-LC", "pPS\n42 h", 0),
    ("42h", "aPS-LC", "aPS\n42 h", .5),
    ("72h", "pPS-LC", "pPS\n72 h", 1),
    ("72h", "aPS-LC", "aPS\n72 h", 1.5),
    ("72h", "NasM",   "NasM\n72 h", 2),
]
# ---------------------------------------------------------------------------
# Summarize the replicate-specific KD − Ctrl effects
# ---------------------------------------------------------------------------

plot_rows = []

for timepoint, state, label, x in PLOT_GROUPS:

    values = wide.loc[
        (
            wide["timepoint_true"].astype(str)
            == timepoint
        )
        &
        (
            wide["state_plot"].astype(str)
            == state
        ),
        "KD_minus_Ctrl",
    ].dropna()

    if len(values) == 0:
        print(f"No values found for {timepoint}, {state}")
        continue

    plot_rows.append({
        "timepoint": timepoint,
        "state": state,
        "label": label,
        "x": x,
        "mean": values.mean(),
        "minimum": values.min(),
        "maximum": values.max(),
        "n_repeats": len(values),
    })

plot_df = pd.DataFrame(plot_rows)

display(plot_df)


# ---------------------------------------------------------------------------
# Asymmetric min–max distances around the mean
# ---------------------------------------------------------------------------

means = plot_df["mean"].to_numpy()

yerr = np.vstack([
    means - plot_df["minimum"].to_numpy(),
    plot_df["maximum"].to_numpy() - means,
])


# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

FIG_C = "#D27D72"

fig, ax = plt.subplots(
    figsize=(3.6, 2.6)
)

ax.errorbar(
    plot_df["x"],
    means,
    yerr=yerr,

    fmt="o",
    markersize=7,

    color=FIG_C,
    markerfacecolor=FIG_C,
    markeredgecolor="white",
    markeredgewidth=0.8,

    ecolor=FIG_C,
    elinewidth=1.2,
    capsize=3,
    capthick=1.2,

    zorder=3,
)


# Zero indicates no KD effect
ax.axhline(
    0,
    color="#999999",
    linewidth=0.9,
    zorder=0,
)



# ---------------------------------------------------------------------------
# Axis labels
# ---------------------------------------------------------------------------

ax.set_xticks(
    plot_df["x"]
)

tick_texts = ax.set_xticklabels(
    plot_df["label"],
    fontsize=14,
)

for text in tick_texts:
    text.set_linespacing(1.0)

ax.set_ylabel(
    "KD − Ctrl module score",
    fontsize=14,
)


# ---------------------------------------------------------------------------
# Styling
# ---------------------------------------------------------------------------

ax.grid(False)

ax.tick_params(
    axis="both",
    length=0,
)

for spine in [
    "top",
    "right",
]:
    ax.spines[spine].set_visible(False)

ax.margins(
    x=0.08,
    y=0.15,
)

fig.tight_layout()


# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------

for ext in SAVE_FORMATS:

    fig.savefig(
        FIG
        / f"KD_APLNR_module_mean_range.{ext}",
        bbox_inches="tight",
    )

plt.show()

In [ ]:
# ============================================================================
# TOP CONTROL-DEFINED APLNR-MODULE HUB GENES
#
# Rows:
#   Top module genes selected by Ctrl-defined WGCNA kME
#
# Columns:
#   Mesoderm state x time
#
# Color:
#   Mean KD - Ctrl gene effect across biological repeats
#
# Faded cells with x:
#   Chicago and Michigan effects have opposite directions
#
# Requires:
#   module_def
#   gene_delta_summary
#   FIG
#   SAVE_FORMATS
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle


# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------

N_HUB_GENES = 40

PLOT_GROUPS = [
    ("42h", "pPS-LC", "pPS\n42 h"),
    ("42h", "aPS-LC", "aPS\n42 h"),
    ("72h", "pPS-LC", "pPS\n72 h"),
    ("72h", "aPS-LC", "aPS\n72 h"),
    ("72h", "NasM",   "NasM\n72 h"),
]


# ------------------------------------------------------------------
# Identify the kME column
# ------------------------------------------------------------------

kme_candidates = [
    col
    for col in module_def.columns
    if "kme" in col.lower()
]

if not kme_candidates:
    raise ValueError(
        "No kME column found in module_def. Columns are:\n"
        + "\n".join(module_def.columns.astype(str))
    )

# Prefer an APLNR-module-specific kME column if several exist
preferred = [
    col
    for col in kme_candidates
    if "black" in col.lower()
]

KME_COL = (
    preferred[0]
    if preferred
    else kme_candidates[0]
)

print("Using kME column:", KME_COL)


# ------------------------------------------------------------------
# Prepare control-defined module ranking
# ------------------------------------------------------------------

hub_table = (
    module_def[
        ["gene", KME_COL]
    ]
    .copy()
)

hub_table["gene"] = (
    hub_table["gene"]
    .astype(str)
)

hub_table[KME_COL] = pd.to_numeric(
    hub_table[KME_COL],
    errors="coerce",
)

hub_table = (
    hub_table
    .dropna(subset=[KME_COL])
    .query("gene != 'APLNR'")
    .assign(
        abs_kME=lambda x:
            x[KME_COL].abs()
    )
    .sort_values(
        "abs_kME",
        ascending=False,
    )
    .drop_duplicates(
        "gene",
        keep="first",
    )
)


# ------------------------------------------------------------------
# Prepare gene-level KD effects
# ------------------------------------------------------------------

effect = gene_delta_summary.copy()

effect["timepoint_true"] = (
    effect["timepoint_true"]
    .astype(str)
)

effect["state_plot"] = (
    effect["state_plot"]
    .astype(str)
)

effect["plot_key"] = (
    effect["timepoint_true"]
    + "__"
    + effect["state_plot"]
)

plot_keys = [
    f"{timepoint}__{state}"
    for timepoint, state, _ in PLOT_GROUPS
]

plot_labels = [
    label
    for _, _, label in PLOT_GROUPS
]

effect = effect[
    effect["plot_key"].isin(plot_keys)
].copy()


# ------------------------------------------------------------------
# Restrict selection only by data availability
#
# This does NOT select genes based on their KD effect or agreement.
# It requires both repeats to be available for all displayed states.
# ------------------------------------------------------------------

complete_genes = (
    effect[
        effect["n_sites"] == 2
    ]
    .groupby(
        "gene",
        observed=True,
    )["plot_key"]
    .nunique()
)

complete_genes = (
    complete_genes[
        complete_genes == len(plot_keys)
    ]
    .index
)

hub_table = hub_table[
    hub_table["gene"].isin(
        complete_genes
    )
].copy()

selected_genes = (
    hub_table
    .head(N_HUB_GENES)["gene"]
    .tolist()
)

print(
    f"Selected {len(selected_genes)} "
    f"Ctrl-defined hub genes"
)

if len(selected_genes) < N_HUB_GENES:
    print(
        "Fewer than requested because some hub genes "
        "were not measured in both repeats for every state."
    )


# ------------------------------------------------------------------
# Mean-effect and replicate-agreement matrices
# ------------------------------------------------------------------

plot_effect = effect[
    effect["gene"].isin(
        selected_genes
    )
].copy()

mean_matrix = (
    plot_effect
    .pivot(
        index="gene",
        columns="plot_key",
        values="mean_delta",
    )
    .reindex(
        index=selected_genes,
        columns=plot_keys,
    )
)

agreement_matrix = (
    plot_effect
    .pivot(
        index="gene",
        columns="plot_key",
        values="reproduced",
    )
    .reindex(
        index=selected_genes,
        columns=plot_keys,
    )
    .fillna(False)
    .astype(bool)
)

mean_matrix.columns = plot_labels
agreement_matrix.columns = plot_labels


# ------------------------------------------------------------------
# Symmetric color range
#
# Use the 95th percentile so one extreme gene does not flatten the rest.
# Values beyond the limit remain saturated.
# ------------------------------------------------------------------

values = mean_matrix.to_numpy(dtype=float)

vmax = np.nanpercentile(
    np.abs(values),
    95,
)

vmax = max(
    vmax,
    0.5,
)

print(
    f"Heatmap color range: "
    f"{-vmax:.2f} to {vmax:.2f}"
)


# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------

fig_height = max(
    5.8,
    0.20 * len(selected_genes),
)

fig, ax = plt.subplots(
    figsize=(5.4, fig_height)
)

sns.heatmap(
    mean_matrix,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-vmax,
    vmax=vmax,

    linewidths=0.35,
    linecolor="white",

    cbar_kws={
        "label": "Mean KD − Ctrl\ngene effect",
        "shrink": 0.55,
        "pad": 0.03,
    },
)


# ------------------------------------------------------------------
# Fade and mark cells where repeat directions disagree
# ------------------------------------------------------------------

for row_i, gene in enumerate(
    agreement_matrix.index
):

    for col_i, column in enumerate(
        agreement_matrix.columns
    ):

        agrees = agreement_matrix.loc[
            gene,
            column,
        ]

        if not agrees:

            ax.add_patch(
                Rectangle(
                    (col_i, row_i),
                    1,
                    1,
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.68,
                    zorder=3,
                )
            )

            ax.text(
                col_i + 0.5,
                row_i + 0.5,
                "×",
                ha="center",
                va="center",
                fontsize=7,
                color="#707070",
                zorder=4,
            )


# ------------------------------------------------------------------
# Styling
# ------------------------------------------------------------------

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_title(
    "Control-defined APLNR-module hub genes",
    fontsize=12,
    pad=10,
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0,
    fontsize=9,
)

for label in ax.get_xticklabels():
    label.set_linespacing(0.75)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=8,
)

ax.tick_params(
    axis="both",
    length=0,
)

ax.text(
    1.0,
    -0.075,
    "×  opposite directions between repeats",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    color="#606060",
)

fig.tight_layout()


# ------------------------------------------------------------------
# Save plotted data and figure
# ------------------------------------------------------------------

heatmap_table = (
    mean_matrix
    .reset_index()
    .rename(
        columns={
            "index": "gene",
        }
    )
)

heatmap_table.to_csv(
    MODULE_OUT
    / "APLNR_module_top_hub_gene_mean_effect_heatmap.csv",
    index=False,
)

agreement_matrix.reset_index().to_csv(
    MODULE_OUT
    / "APLNR_module_top_hub_gene_repeat_agreement.csv",
    index=False,
)

for ext in SAVE_FORMATS:

    fig.savefig(
        FIG
        / f"KD_APLNR_module_top{len(selected_genes)}_hub_heatmap.{ext}",
        bbox_inches="tight",
    )

plt.show()

### Supplementary - genome-wide concordance scatter

In [ ]:
# ============================================================
# SUPPLEMENTARY — genome-wide cross-repeat concordance scatter
# Superseded as a main panel by the forest plot below, which makes the same
# point (the two repeats agree) on the genes you actually discuss instead of
# via 20,000 grey dots. Keep this only if you want a genome-wide statement:
# after centring, sign agreement is 70.6%, and 490 genes clear |log2FC|>0.5
# in both repeats.
# Reads the canonical pseudobulk output from kd_pseudobulk.py.
# It no longer builds its own CPM pseudobulk: the per-repeat log2FCs in that file are
# computed on DESeq2 median-of-ratios normalised counts and MEDIAN-CENTRED. Centring
# matters -- raw CPM log2FCs on this data sit -0.285 below zero genome-wide because KD
# cells are 25-38% less complex, which made 76% of all genes negative in both repeats
# and sign agreement nearly automatic (4,164 genes flagged "reproducible"). Centred, the
# flag means what it says.
# Run kd_pseudobulk.py first if KD_pseudobulk_results.csv is missing.
# ============================================================
TISSUE = "all_mesoderm"                      # or "NasM" for the nascent-mesoderm-only view
pb = pd.read_csv(TAB/"KD_pseudobulk_results.csv", index_col=0)
pb = pb[pb["tissue"] == TISSUE]
sites = ["Michigan", "Chicago"]
L = pb[[f"log2FC_{s}" for s in sites]].rename(columns=lambda c: c.replace("log2FC_", ""))
L["mean"], L["agree"] = L.mean(1), pb["reproducible"].to_numpy()

KEY = ["APLNR","APELA","WNT8A","TBX6","MSGN1","DLL3","MESP1","MESP2","EOMES","TBXT",
       "LEF1","AXIN2","SP5","FOXC2","MEOX1","CDX2"]
lab  = {g for g in KEY if g in L.index}
lab |= set(L[L.agree].sort_values("mean").head(10).index)     # top reproducible DOWN
lab |= set(L[L.agree].sort_values("mean").tail(6).index)      # top reproducible UP

lim = float(np.nanpercentile(np.abs(L[sites].values), 99)) * 1.15
fig, ax = plt.subplots(figsize=(6.8, 6.8))
ax.axhline(0, color="#c8c8c8", lw=.8); ax.axvline(0, color="#c8c8c8", lw=.8)
ax.plot([-lim, lim], [-lim, lim], color="#e0e0e0", lw=1, zorder=0)
ax.scatter(L["Michigan"], L["Chicago"], s=10,
           c=np.where(L["agree"], "#C0563E", "#CBC9C4"), alpha=.55, linewidths=0)
for g in lab:
    if g not in L.index: continue
    ax.annotate(g, (L.loc[g,"Michigan"], L.loc[g,"Chicago"]), fontsize=8, fontweight="bold",
                color="#7A2E1E" if L.loc[g,"agree"] else "#8a8a8a",
                xytext=(3,3), textcoords="offset points")
ax.set_xlabel("log2FC  Michigan  (KD/Ctrl, centred)", fontsize=11)
ax.set_ylabel("log2FC  Chicago  (KD/Ctrl, centred)", fontsize=11)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
if SHOW_TITLES:
    ax.set_title(f"{TISSUE} KD vs Ctrl, 72h — biological repeats\n"
                 f"red = reproducible ({int(L.agree.sum()):,} of {len(L):,} genes)", fontsize=12)
for sp in ax.spines.values(): sp.set_visible(False)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"S_mesoderm_concordance.{ext}", bbox_inches="tight")
L.to_csv(TAB/"D_mesoderm_concordance.csv")
plt.show()


### D - Panel gene effect sizes, both biological repeats

In [ ]:
# ---- Panel: pre-specified gene panel, effect size with BOTH biological repeats ----
# Replaces the genome-wide concordance scatter as the main gene-level readout.
# The scatter made one point ("the repeats agree") using 20,000 grey dots; this makes the
# same point per gene, on the genes you actually discuss, and adds the effect size.
#   diamond = DESeq2 shrunk log2FC   circle = Chicago   square = Michigan
#   filled/red = panel-FDR < 0.05
pb    = pd.read_csv(TAB/"KD_pseudobulk_results.csv", index_col=0)
panel = pd.read_csv(TAB/"KD_pseudobulk_panel_tests.csv", index_col=0)
TISSUES = ["all_mesoderm", "NasM"]

fig, axes = plt.subplots(1, len(TISSUES), figsize=(5.6*len(TISSUES), 7.2), sharex=True)
for ax, tis in zip(np.atleast_1d(axes), TISSUES):
    d = pb[pb["tissue"] == tis]
    pt = panel[panel["tissue"] == tis]
    genes = [g for g in pt.index if g in d.index]
    d = d.loc[genes].assign(q=pt.loc[genes, "panel_padj"])
    d = d.sort_values("log2FC_shrunk")
    y = np.arange(len(d))
    sig = (d["q"] < 0.05).to_numpy()

    ax.axvline(0, color="#B8B5AE", lw=1, zorder=0)
    for yi, s in zip(y, sig):
        ax.hlines(yi, -3, 3, color="#F6F4F1" if yi % 2 else "white", lw=14, zorder=-1)
    ax.hlines(y, d["log2FC_Chicago"], d["log2FC_Michigan"],
              color=np.where(sig, "#E0BEB4", "#D8D5D0"), lw=2, zorder=1)
    ax.scatter(d["log2FC_Chicago"],  y, marker="o", s=38, facecolor="white",
               edgecolor=np.where(sig, "#C0563E", "#A9A6A0"), lw=1.4, zorder=3, label="Chicago")
    ax.scatter(d["log2FC_Michigan"], y, marker="s", s=38, facecolor="white",
               edgecolor=np.where(sig, "#C0563E", "#A9A6A0"), lw=1.4, zorder=3, label="Michigan")
    ax.scatter(d["log2FC_shrunk"],   y, marker="D", s=52,
               color=np.where(sig, "#C0563E", "#8A8780"), zorder=4, label="shrunk estimate")
    ax.set_yticks(y)
    ax.set_yticklabels([f"$\\bf{{{g}}}$" if s else g for g, s in zip(d.index, sig)], fontsize=10)
    ax.set_xlabel("log2FC  KD / Ctrl")
    ax.set_title(f"{tis}   ({int(sig.sum())}/{len(d)} at panel-FDR<0.05)", fontsize=11)
    ax.set_ylim(-0.8, len(d)-0.2)
    for sp in ["top", "right", "left"]: ax.spines[sp].set_visible(False)
    ax.tick_params(left=False)
np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("Pre-specified panel: KD effect and cross-repeat agreement", fontsize=12, y=0.99)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"D_panel_forest.{ext}", bbox_inches="tight")
plt.show()


## E — migration / guidance programme (both sites)

In [ ]:
guid = ["ROBO1","ROBO2","SLIT1","SLIT2","PLXNA2","PLXNA4","SEMA3A","SEMA6A",
        "EPHA4","EPHB2","NRP1","NRP2","CDH2","CDH11","SPARC","FN1","ITGA5","ITGB5"]
for st in ["aPS-LC","pPS-LC"]:
    d = pd.read_csv(TAB/f"gw_decompose_{st}_72h.csv")
    g = d[(d["gene"].isin(guid)) & (d["agree"]==True)].sort_values("dfrac")
    print(f"\n=== {st} (both sites agree) ===")
    print(g[["gene","dfrac","dlvl"]].round(2).to_string(index=False) if len(g)
          else "  none passed the both-sites filter")

## Methods

Cells restricted to the Stan et al. filtered set (per-sample library-size gating); the same set used for Figure 2. The knockdown marker and composition effects are unchanged whether or not this filter is applied. Composition computed as within-sample fractions. n=2 independent differentiations (Michigan, Chicago); reported evidence is consistent direction across both sites. Gene-level differential expression is reported from bulk RNA-seq (Figure 6); this figure reports cell-type composition and relative marker expression, which single-cell uniquely provides. Timepoints 42h and 72h.